## 1 · Setup & configuration

In [1]:
from __future__ import annotations
import gc
import hashlib
import json
import math
import os
import platform
import re
import sys
import time
import unicodedata
import warnings
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

try:
    from IPython.display import display
except ImportError:  # plain python
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

SEED = 42
np.random.seed(SEED)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
_raw_candidates = [
    os.environ.get("RAW_TRANSCRIPTS_PATH"),
    PROJECT_ROOT / "data" / "final_transcripts.json",
    PROJECT_ROOT / "final_transcripts.json",
    Path.cwd() / "final_transcripts.json",
]
RAW_PATH = next((Path(p) for p in _raw_candidates if p and Path(p).is_file()), None)
if RAW_PATH is None:
    raise FileNotFoundError(
        "final_transcripts.json not found. Put it in data/ (or the project root) "
        "or set the RAW_TRANSCRIPTS_PATH environment variable."
    )
OUT_DIR = Path(os.environ.get("CLEANED_DATA_DIR") or (PROJECT_ROOT / "cleaned_data"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

DROP_EXACT_DUPLICATES = True       
NEAR_DUP_DETECTION = True          
NEAR_DUP_THRESHOLD = 0.90          
TEMPLATE_JACCARD = 0.50           
TEMPLATE_MAX_GROUP_SHARE = 0.01    
OPENING_TURNS = 4                  
LEAKAGE_AUC_FLAG = 0.75           
LEAKAGE_EXCESS_AUC = 0.05          
LEAKAGE_CONTENT_SHARE = 0.05       
SPLIT_FRACTIONS = (0.80, 0.10, 0.10)   
SPLIT_SALT = "crest-hackathon-v1"  
ROLE_CLF_MAX_TRAIN_TURNS = 200_000 
ROLE_MIN_MARGIN = 2.0              
REPEAT_SIM_THRESHOLD = 0.50        
MIN_TURNS_OK = 4                   
DOMAIN_INTENT_PURITY = 0.75       
DOMAIN_OVERRIDES: dict[str, str] = {} 

print(f"Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | scikit-learn {sklearn.__version__}")
print("RAW_PATH :", RAW_PATH.resolve())
print("OUT_DIR  :", OUT_DIR.resolve())
T0 = time.perf_counter()
REPORT: dict = {"config": {
    "DROP_EXACT_DUPLICATES": DROP_EXACT_DUPLICATES, "NEAR_DUP_DETECTION": NEAR_DUP_DETECTION,
    "NEAR_DUP_THRESHOLD": NEAR_DUP_THRESHOLD, "TEMPLATE_JACCARD": TEMPLATE_JACCARD,
    "TEMPLATE_MAX_GROUP_SHARE": TEMPLATE_MAX_GROUP_SHARE, "OPENING_TURNS": OPENING_TURNS,
    "LEAKAGE_AUC_FLAG": LEAKAGE_AUC_FLAG, "LEAKAGE_EXCESS_AUC": LEAKAGE_EXCESS_AUC,
    "LEAKAGE_CONTENT_SHARE": LEAKAGE_CONTENT_SHARE, "SPLIT_FRACTIONS": SPLIT_FRACTIONS, "SPLIT_SALT": SPLIT_SALT,
    "ROLE_CLF_MAX_TRAIN_TURNS": ROLE_CLF_MAX_TRAIN_TURNS, "ROLE_MIN_MARGIN": ROLE_MIN_MARGIN,
    "REPEAT_SIM_THRESHOLD": REPEAT_SIM_THRESHOLD, "MIN_TURNS_OK": MIN_TURNS_OK,
    "DOMAIN_INTENT_PURITY": DOMAIN_INTENT_PURITY, "DOMAIN_OVERRIDES": DOMAIN_OVERRIDES, "SEED": SEED}}
QC_LOG: list[dict] = []


def qc(tid, stage: str, issue: str, detail: str = "") -> None:
    QC_LOG.append({"transcript_id": tid, "stage": stage, "issue": issue, "detail": str(detail)[:300]})

Python 3.13.5 | pandas 3.0.6 | numpy 2.5.3 | scikit-learn 1.9.1
RAW_PATH : D:\Projects & Research\NLP_Agentic_Inter_IIT\causal-rationale-hackathon\data\final_transcripts.json
OUT_DIR  : D:\Projects & Research\NLP_Agentic_Inter_IIT\causal-rationale-hackathon\cleaned_data


## 2 · Load the raw file & audit its schema

In [2]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def load_raw(path: Path) -> list:
    text = path.read_text(encoding="utf-8-sig")
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:  
        obj = [json.loads(line) for line in text.splitlines() if line.strip()]
    if isinstance(obj, dict):
        for key in ("data", "records", "transcripts", "conversations"):
            if isinstance(obj.get(key), list):
                obj = obj[key]
                break
    if not isinstance(obj, list):
        raise ValueError(f"Unexpected top-level JSON type: {type(obj).__name__}")
    return obj


raw = load_raw(RAW_PATH)
RAW_SHA256 = sha256_file(RAW_PATH)
print(f"records: {len(raw):,} | sha256: {RAW_SHA256[:16]}…")

EXPECTED_KEYS = ["transcript_id", "time_of_interaction", "domain", "intent", "reason_for_call", "conversation"]
key_counter, null_counter, type_counter = Counter(), Counter(), defaultdict(Counter)
turn_key_counter, turn_issue_counter = Counter(), Counter()
for rec in raw:
    if not isinstance(rec, dict):
        continue
    for k, v in rec.items():
        key_counter[k] += 1
        type_counter[k][type(v).__name__] += 1
        if v is None or (isinstance(v, str) and not v.strip()):
            null_counter[k] += 1
    for t in rec.get("conversation") or []:
        if isinstance(t, dict):
            turn_key_counter.update(t.keys())
            if not isinstance(t.get("speaker"), str):
                turn_issue_counter["speaker_not_str"] += 1
            if not isinstance(t.get("text"), str):
                turn_issue_counter["text_not_str"] += 1
            elif not t["text"].strip():
                turn_issue_counter["text_empty"] += 1
        else:
            turn_issue_counter["turn_not_dict"] += 1

schema_audit = pd.DataFrame([
    {"field": k, "present": key_counter[k], "missing_key": len(raw) - key_counter[k],
     "null_or_empty": null_counter[k], "types": dict(type_counter[k])}
    for k in list(dict.fromkeys(EXPECTED_KEYS + list(key_counter)))
])
display(schema_audit)
print("turn keys:", dict(turn_key_counter), "| turn issues:", dict(turn_issue_counter) or "none")
REPORT["schema_audit"] = schema_audit.astype(str).to_dict("records")
REPORT["turn_issues"] = dict(turn_issue_counter)

records: 19,621 | sha256: 34ba8faec8f8c1a7…


,field,present,missing_key,null_or_empty,types
0,transcript_id,19621,0,0,{'str': 19621}
1,time_of_interaction,19621,0,0,{'str': 19621}
2,domain,19621,0,0,{'str': 19621}
3,intent,19621,0,0,{'str': 19621}
4,reason_for_call,19621,0,0,{'str': 19621}
5,conversation,19621,0,0,{'list': 19621}


turn keys: {'speaker': 686764, 'text': 686764} | turn issues: {'text_empty': 60}


### 2.1 Coerce records and resolve duplicate ids

In [3]:
def raw_signature(turns: list[dict]) -> str:
    payload = "\n".join(f"{t['speaker']}|{t['text']}" for t in turns)
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()


records: list[dict] = []
seen_ids: dict[str, str] = {}      # id -> raw signature of its first occurrence
dup_id_counter: Counter = Counter()
for i, rec in enumerate(raw):
    if not isinstance(rec, dict):
        qc(None, "load", "record_not_dict", f"index {i}")
        continue
    tid = rec.get("transcript_id")
    tid = str(tid).strip() if tid is not None else ""
    if not tid:
        tid = f"missing-id-{i:06d}"
        qc(tid, "load", "missing_transcript_id", f"index {i}")
    conv = rec.get("conversation")
    if not isinstance(conv, list):
        qc(tid, "load", "conversation_not_list", type(conv).__name__)
        conv = []
    turns = []
    for j, t in enumerate(conv):
        if not isinstance(t, dict):
            qc(tid, "load", "turn_not_dict", f"turn {j}")
            t = {"speaker": "", "text": str(t)}
        spk, txt = t.get("speaker"), t.get("text")
        turns.append({"orig_index": j,
                      "speaker": "" if spk is None else str(spk),
                      "text": "" if txt is None else str(txt)})
    sig = raw_signature(turns)
    if tid in seen_ids:
        if seen_ids[tid] == sig:
            qc(tid, "load", "duplicate_id_identical_content_dropped", f"index {i}")
            continue
        dup_id_counter[tid] += 1
        new_tid = f"{tid}__dup{dup_id_counter[tid]}"
        qc(new_tid, "load", "duplicate_id_different_content_renamed", f"original id {tid}")
        tid = new_tid
    seen_ids[tid] = sig
    meta = {k: rec.get(k) for k in ("time_of_interaction", "domain", "intent", "reason_for_call")}
    extra = {k: v for k, v in rec.items() if k not in EXPECTED_KEYS}
    records.append({"transcript_id": tid, "file_order": i, **meta, "raw_turns": turns, "extra_meta": extra})

REPORT["n_raw_records"] = len(raw)
REPORT["n_records_after_load"] = len(records)
del raw         
gc.collect()
REPORT["n_raw_turns"] = sum(len(r["raw_turns"]) for r in records)
print(f"records kept: {len(records):,} | raw turns: {REPORT['n_raw_turns']:,} | "
      f"duplicate ids renamed: {sum(dup_id_counter.values())}")

records kept: 19,621 | raw turns: 686,764 | duplicate ids renamed: 0


## 3 · Text & label utilities (with self-tests)

In [4]:
QUOTE_MAP = str.maketrans({
    "\u2018": "'", "\u2019": "'", "\u201a": "'", "\u201b": "'", "\u2032": "'", "\u00b4": "'", "`": "'",
    "\u201c": '"', "\u201d": '"', "\u201e": '"', "\u2033": '"',
    "\u2013": "-", "\u2014": "-", "\u2212": "-", "\u00a0": " ",
})
ZERO_WIDTH_RE = re.compile(r"[\u200b-\u200f\u202a-\u202e\u2060-\u2064\ufeff]")
CONTROL_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
LITERAL_ESCAPE_RE = re.compile(r"\\[nrt]")
WS_RE = re.compile(r"\s+")
WORD_RE = re.compile(r"[a-z0-9]+(?:'[a-z]+)?")
STRAY_MARKUP_RE = re.compile(r"[*\[\]<>{}|]")


def normalize_text(s: str) -> str:
    s = unicodedata.normalize("NFKC", s)
    s = s.translate(QUOTE_MAP)
    s = ZERO_WIDTH_RE.sub("", s)
    s = LITERAL_ESCAPE_RE.sub(" ", s)
    s = CONTROL_RE.sub(" ", s)
    s = s.lower()
    s = re.sub(r"\buh-huh\b", "uh huh", s)
    s = re.sub(r"\b(?:mm-hmm|mm-hm|mhm-hmm|um-hum|mm-mm)\b", "mm hmm", s)
    return WS_RE.sub(" ", s).strip()

FILLED_PAUSE_RE = re.compile(r"\b(?:u+m+|u+h+m*(?!\s+huh\b)|e+r+m+|h+m+|a+h+)\b")
NUMBER_WORDS = set(("zero oh one two three four five six seven eight nine ten eleven twelve thirteen fourteen "
                    "fifteen sixteen seventeen eighteen nineteen twenty thirty forty fifty sixty seventy "
                    "eighty ninety hundred thousand double triple").split())
REP_KEEP = {"bye", "very", "really", "ha", "haha", "hey", "hi", "no", "yes", "yeah", "okay", "ok", "so", "bla"}
REP_BIGRAM_RE = re.compile(r"\b([a-z']+ [a-z']+)(?: \1\b)+")
REP_WORD_RE = re.compile(r"\b([a-z']+)(?: \1\b)+(?! (?:know|mean)\b)")
HEDGE_RE = re.compile(r"\b(?:i guess|i suppose|maybe|perhaps|probably|i'm not sure|i am not sure|not sure|"
                      r"i don't know|i do not know|kind of|sort of|i think)\b")
DISCOURSE_MARKER_RE = re.compile(r"\b(?:you know|i mean)\b")


def _is_protected(token_or_bigram: str) -> bool:
    return any(w in NUMBER_WORDS or w.isdigit() or (len(w) == 1 and w not in ("i", "a"))
               for w in token_or_bigram.split())


def _collapse(m: re.Match) -> str:
    return m.group(0) if _is_protected(m.group(1)) else m.group(1)


def count_filled_pauses(text: str) -> int:
    return len(FILLED_PAUSE_RE.findall(text))


def count_repetitions(text: str) -> int:
    n = sum(1 for m in REP_WORD_RE.finditer(text) if m.group(1) not in REP_KEEP and not _is_protected(m.group(1)))
    return n + sum(1 for m in REP_BIGRAM_RE.finditer(text) if not _is_protected(m.group(1)))


def make_clean_text(text: str) -> str:
    t = FILLED_PAUSE_RE.sub(" ", text)
    t = WS_RE.sub(" ", t).strip()
    t = REP_BIGRAM_RE.sub(_collapse, t)
    t = REP_WORD_RE.sub(_collapse, t)
    t = WS_RE.sub(" ", t).strip()
    return t if t else text 

_PAREN_STAGE_WORDS = (r"(?:long |brief |short |slight |a )?(?:pause[sd]?|pausing|silence|sighs?|sighing|laugh(?:s|ing|ter)?|"
                      r"chuckl(?:e|es|ing)|inaudible|unintelligible|cross ?talk|background noise|noise|static|"
                      r"hold music|music|on hold|holding|typing|keyboard(?: typing)?|clears throat|cough(?:s|ing)?|"
                      r"after a (?:moment|pause|while|beat|few seconds)|a moment later|moments later|transferr?ing|"
                      r"transfers?|call (?:ended|ends|disconnected|drops|dropped)|end of call|beep|ringing|dial tone)")
STAGE_RES = [
    re.compile(r"\*\s*([^*]{1,80}?)\s*\*"),                 
    re.compile(r"\[\s*([^\]]{1,200}?)\s*\]"),                
    re.compile(r"<\s*([^>]{1,80}?)\s*>"),                    
    re.compile(r"\(\s*(" + _PAREN_STAGE_WORDS + r"[^)]{0,40}?)\s*\)"), 
]
NONVERBAL_TYPES = [
    ("CALL_END", re.compile(r"\b(?:call (?:ended|ends|end|disconnected|dropped|drops|terminated)|end of (?:the )?(?:call|transcript|conversation|recording)|hangs? up|hung up|disconnect(?:ed|s)?)\b")),
    ("TRANSFER", re.compile(r"\btransfer")),
    ("HOLD", re.compile(r"\b(?:hold|holding|music)\b")),
    ("PAUSE", re.compile(r"\b(?:pause[sd]?|pausing|silence|silent|moments?|beat|wait(?:ing)?|later|while|seconds)\b")),
    ("SIGH", re.compile(r"\bsigh")),
    ("LAUGH", re.compile(r"\b(?:laugh|chuckl|giggl)")),
    ("NOISE", re.compile(r"\b(?:inaudible|unintelligible|cross ?talk|noise|static|garbled|asr|unclear|mumbl|beep|ring|cough|clears throat|dial tone)")),
    ("TYPING", re.compile(r"\b(?:typing|keyboard|click)")),
    ("NOTE", re.compile(r"^(?:system note|note|notes|summary|system|agent note|annotator|comment|analysis|key events?|outcome|disposition|survey|tag|label|sentiment)\b")),
]
CALL_END_TEXT_RE = re.compile(
    r"^(?:the )?(?:call (?:ended|ends|end|disconnected|dropped|terminated|concluded)|end of (?:the )?(?:call|transcript|conversation|recording)|"
    r"(?:customer|caller|agent) (?:hung up|hangs up|disconnected)|transcript ends?|conversation ended|line disconnected)\W*$")


def classify_stage(content: str) -> str:
    c = content.lower().strip()
    for name, rx in NONVERBAL_TYPES:
        if rx.search(c):
            return name
    return "OTHER"


def extract_stage_directions(text: str):
    events, notes = [], []

    def _grab(m: re.Match) -> str:
        content = m.group(1).strip()
        kind = classify_stage(content)
        if kind == "NOTE":
            notes.append(content)
        else:
            events.append({"type": kind, "raw": content})
        return " "

    for rx in STAGE_RES:
        text = rx.sub(_grab, text)
    text = STRAY_MARKUP_RE.sub(" ", text)
    return WS_RE.sub(" ", text).strip(), events, notes

_EMB_SPK = r"agent|customer|supervisor|manager|caller|representative|specialist"
EMBED_RE = re.compile(
    r"(?:\\n|\\r|\n|\r)+\s*(?:(?P<num>\d{1,3})\s*[.):\-]?\s*(?P<spk1>" + _EMB_SPK + r")\b\s*:?|(?P<spk2>" + _EMB_SPK + r")\s*:)\s*",
    re.IGNORECASE)

def split_embedded(text: str) -> list[tuple[str | None, str]]:
    matches = list(EMBED_RE.finditer(text))
    if not matches:
        return [(None, text)]
    pieces = []
    head = text[:matches[0].start()]
    if head.strip():
        pieces.append((None, head))
    for k, m in enumerate(matches):
        end = matches[k + 1].start() if k + 1 < len(matches) else len(text)
        seg = text[m.end():end]
        spk = (m.group("spk1") or m.group("spk2")).capitalize()
        if seg.strip():
            pieces.append((spk, seg))
    return pieces

ANNOTATION_PREFIX_CHARS = tuple("-[*(#<{>=•")
ANNOTATION_LABELS = {
    "system note", "note", "notes", "call notes", "additional note", "domain", "domain category", "outcome",
    "tracker", "listener", "teacher", "student", "key event", "key events", "key events summary", "summary",
    "call summary", "summary of key events", "end of call summary", "call disposition", "disposition",
    "survey code", "wrap-up notes", "wrap up notes", "patterns noted", "please advise", "asr noise", "narrator",
    "transcript", "context", "scenario", "setting", "background", "category", "intent", "label", "tag",
    "metadata", "annotation", "comment", "observation", "analysis", "call outcome", "call result", "result",
    "resolution", "sentiment", "timestamp", "action items", "next steps"}
ANNOTATION_LABEL_RE = re.compile(r"\b(?:notes?|summary|key events?|disposition|survey code|wrap-?up|patterns noted|"
                                 r"please advise|metadata|annotation|timestamp)\b")
STAGE_PREFIX_RE = re.compile(r"^[\(\[\*]+\s*([^\)\]\*]{1,40}?)\s*[\)\]\*]+\s*(.+)$")
SYSTEM_LABEL_RE = re.compile(r"^(?:system|ivr|automated(?: system| message| voice| assistant)?|bot|chatbot|"
                             r"virtual assistant|voice assistant|ai|recording|auto(?:mated)? attendant)$")
SUPERVISOR_LABEL_RE = re.compile(r"\b(?:supervisor|manager|team lead(?:er)?|escalation|senior (?:agent|representative|advisor))\b")
SPECIALIST_LABEL_RE = re.compile(r"\b(?:specialist|expert|technician|underwriter|adjuster|investigator|officer|"
                                 r"department|analyst|engineer)\b")
AGENT_LABEL_RE = re.compile(r"\b(?:agent|representative|rep|csr|advisor|adviser|associate|operator|consultant|"
                            r"customer (?:service|care|support|success))\b")
CUSTOMER_LABEL_RE = re.compile(r"\b(?:customer|caller|client|user|member|policy ?holder|guest|passenger|"
                               r"account ?holder|cardholder|subscriber|patient|buyer|shopper)\b")
PERSON_LABEL_RE = re.compile(r"^[a-z][a-z'.\-]*(?: [a-z][a-z'.\-]*){0,3}$")
NARRATIVE_RE = re.compile(
    r"^(?:the )?(?:customer|caller|agent|representative|rep|supervisor|call|conversation|client)\b.{0,120}?\b"
    r"(?:was|were|is|seemed|expressed|felt|called|asked|requested|mentioned|decided|ended|concluded|occurred|"
    r"escalated|resolved|provided|explained|confirmed|verified|agreed|offered|reported)\b"
    r"|\b(?:no escalation|escalation (?:occurred|was)|call (?:summary|disposition)|key events?|"
    r"resolution (?:achieved|summary)|sentiment (?:was|score)|churn risk|patterns? noted)\b")


def normalize_label(label: str) -> str:
    s = normalize_text(label).replace("_", " ")
    s = re.sub(r"'s\b", "", s)                    
    s = re.sub(r"[^a-z0-9' .\-]", " ", s)
    return WS_RE.sub(" ", s).strip(" .-'")


def label_rule(norm: str) -> str | None:
    if not norm:
        return None
    if SYSTEM_LABEL_RE.match(norm):
        return "system"
    if SUPERVISOR_LABEL_RE.search(norm):
        return "supervisor"
    if SPECIALIST_LABEL_RE.search(norm):
        return "specialist"
    if AGENT_LABEL_RE.search(norm):
        return "agent"
    if CUSTOMER_LABEL_RE.search(norm):
        return "customer"
    return None


def classify_label(label_raw: str) -> dict:
    s = (label_raw or "").strip()
    out = {"kind": "speaker", "label_clean": s, "label_norm": "", "stage_prefix": None, "rule_role": None, "reason": ""}
    m = STAGE_PREFIX_RE.match(s)
    if m:
        rest_norm = normalize_label(m.group(2))
        if rest_norm and rest_norm not in ANNOTATION_LABELS and not ANNOTATION_LABEL_RE.search(rest_norm) \
                and PERSON_LABEL_RE.match(rest_norm):      # "(After A Moment) Noah" -> speaker Noah + PAUSE
            s = m.group(2).strip()
            out.update(label_clean=s, stage_prefix=m.group(1).strip())
    if out["stage_prefix"] is None and s.startswith(ANNOTATION_PREFIX_CHARS):
        return {**out, "kind": "annotation", "reason": "label_prefix"}
    norm = normalize_label(s)
    out["label_norm"] = norm
    if norm in ANNOTATION_LABELS or ANNOTATION_LABEL_RE.search(norm):
        return {**out, "kind": "annotation", "reason": "annotation_label"}
    out["rule_role"] = label_rule(norm)
    return out

assert normalize_text("I\u2019m  here\\n now\u200b") == "i'm here now"
assert split_embedded("hi there \\n 2 customer hello \\n 3 agent bye") == [(None, "hi there "), ("Customer", "hello "), ("Agent", "bye")]
assert split_embedded("i have 2 agents here") == [(None, "i have 2 agents here")]
assert extract_stage_directions("let me check *transferring call* ok") == ("let me check ok", [{"type": "TRANSFER", "raw": "transferring call"}], [])
assert extract_stage_directions("(pause)")[1][0]["type"] == "PAUSE"
assert extract_stage_directions("my plan (the premium one) is slow")[0] == "my plan (the premium one) is slow"
assert extract_stage_directions("sure [note: customer upset] okay") == ("sure okay", [], ["note: customer upset"])
assert CALL_END_TEXT_RE.match("call ended") and not CALL_END_TEXT_RE.match("the call ended badly because you hung up on me")
assert classify_label("(After A Moment) Noah")["label_clean"] == "Noah" and classify_label("(After A Moment) Noah")["kind"] == "speaker"
for lab in ["- Key Event", "Key Events", "[Call Summary", "(System Note", "*[Note", "Domain", "Domain Category", "Outcome", "Student"]:
    assert classify_label(lab)["kind"] == "annotation", lab
assert classify_label("Customer'S")["rule_role"] == "customer"
assert classify_label("Customer Service Rep")["rule_role"] == "agent"
assert classify_label("Escalation_Specialist")["rule_role"] == "supervisor"
assert classify_label("Fraud Specialist")["rule_role"] == "specialist"
assert classify_label("Manager")["rule_role"] == "supervisor"
assert classify_label("James O'Connor")["rule_role"] is None and classify_label("James O'Connor")["kind"] == "speaker"
assert count_filled_pauses("um i uh think uh huh hmm") == 3
assert make_clean_text("i i um want to to cancel") == "i want to cancel"
assert make_clean_text("it's four four five c c one") == "it's four four five c c one"
assert make_clean_text("um") == "um"
assert count_repetitions("i i was i was going to to go bye bye") == 3
print("text/label utilities: all self-tests passed")

text/label utilities: all self-tests passed


## 4 · Structural cleaning pass

In [5]:
def structural_clean(rec: dict):
    tid = rec["transcript_id"]
    segments = []
    for t in rec["raw_turns"]:
        pieces = split_embedded(t["text"])
        if len(pieces) > 1:
            qc(tid, "structure", "embedded_dialogue_split", f"turn {t['orig_index']} -> {len(pieces)} turns")
        for sub, (lab, txt) in enumerate(pieces):
            segments.append({"orig_index": t["orig_index"], "sub_index": sub,
                             "label_raw": lab if lab is not None else t["speaker"],
                             "from_embedded": len(pieces) > 1, "text_raw": txt if len(pieces) > 1 else t["text"]})
    kept, quarantine, events = [], [], []
    for seg in segments:
        info = classify_label(seg["label_raw"])
        text_norm = normalize_text(seg["text_raw"])
        if info["kind"] == "annotation":
            quarantine.append({"transcript_id": tid, "orig_index": seg["orig_index"], "label_raw": seg["label_raw"],
                               "text": text_norm, "reason": info["reason"]})
            continue
        text, nonverbal, notes = extract_stage_directions(text_norm)
        for note in notes:
            quarantine.append({"transcript_id": tid, "orig_index": seg["orig_index"], "label_raw": seg["label_raw"],
                               "text": note, "reason": "inline_note"})
        if info["stage_prefix"]:
            nonverbal.insert(0, {"type": classify_stage(info["stage_prefix"]), "raw": info["stage_prefix"]})
        if info["rule_role"] == "system" and NARRATIVE_RE.search(text):
            quarantine.append({"transcript_id": tid, "orig_index": seg["orig_index"], "label_raw": seg["label_raw"],
                               "text": text, "reason": "system_narrative"})
            continue
        if not WORD_RE.search(text) or CALL_END_TEXT_RE.match(text):
            if CALL_END_TEXT_RE.match(text):
                nonverbal.append({"type": "CALL_END", "raw": text})
            if nonverbal:
                for ev in nonverbal:
                    events.append({"after_turn_id": len(kept) - 1, "orig_index": seg["orig_index"],
                                   "label_raw": seg["label_raw"], **ev})
                kind = "marker_only"
            else:
                kind = "empty"
            qc(tid, "structure", f"{kind}_turn_removed", f"turn {seg['orig_index']}: {seg['text_raw'][:80]!r}")
            continue
        kept.append({
            "orig_index": seg["orig_index"], "sub_index": seg["sub_index"], "from_embedded": seg["from_embedded"],
            "label_raw": seg["label_raw"], "label_clean": info["label_clean"], "label_norm": info["label_norm"],
            "rule_role": info["rule_role"], "text_raw": seg["text_raw"], "text": text,
            "text_clean": make_clean_text(text), "nonverbal": nonverbal,
        })
    return kept, quarantine, events, len(segments)


t0 = time.perf_counter()
QUARANTINE, n_segments_total = [], 0
for rec in records:
    kept, quar, events, n_seg = structural_clean(rec)
    rec["turns"], rec["events"] = kept, events
    rec["n_annotations_removed"] = len([q for q in quar if q["reason"] != "inline_note"])
    rec["had_embedded_dialogue"] = any(t["from_embedded"] for t in kept)
    QUARANTINE.extend(quar)
    n_segments_total += n_seg
qc_df_tmp = pd.DataFrame(QC_LOG)
struct = {
    "raw_turns": REPORT["n_raw_turns"],
    "extra_turns_from_embedded_split": n_segments_total - REPORT["n_raw_turns"],
    "annotation_turns_quarantined": sum(1 for q in QUARANTINE if q["reason"] != "inline_note"),
    "inline_notes_quarantined": sum(1 for q in QUARANTINE if q["reason"] == "inline_note"),
    "marker_only_turns_to_events": int((qc_df_tmp["issue"] == "marker_only_turn_removed").sum()) if len(qc_df_tmp) else 0,
    "empty_turns_removed": int((qc_df_tmp["issue"] == "empty_turn_removed").sum()) if len(qc_df_tmp) else 0,
    "turns_after_structural_pass": sum(len(r["turns"]) for r in records),
    "conversations_with_embedded_dialogue": sum(r["had_embedded_dialogue"] for r in records),
    "conversations_with_annotations": sum(r["n_annotations_removed"] > 0 for r in records),
}
assert struct["raw_turns"] + struct["extra_turns_from_embedded_split"] - struct["annotation_turns_quarantined"] \
    - struct["marker_only_turns_to_events"] - struct["empty_turns_removed"] == struct["turns_after_structural_pass"], struct
REPORT["structural_pass"] = struct
print(json.dumps(struct, indent=2))
print(f"({time.perf_counter() - t0:.1f}s)")
quar_df = pd.DataFrame(QUARANTINE, columns=["transcript_id", "orig_index", "label_raw", "text", "reason"])
print("\nQuarantined labels (top 25):")
print(quar_df["label_raw"].value_counts().head(25).to_string() if len(quar_df) else "none")

{
  "raw_turns": 686764,
  "extra_turns_from_embedded_split": 79,
  "annotation_turns_quarantined": 74,
  "inline_notes_quarantined": 10,
  "marker_only_turns_to_events": 509,
  "empty_turns_removed": 64,
  "turns_after_structural_pass": 686196,
  "conversations_with_embedded_dialogue": 4,
  "conversations_with_annotations": 56
}
(29.9s)

Quarantined labels (top 25):
label_raw
Agent                                                        9
- Key Event                                                  5
Note                                                         4
Key Events                                                   4
(System Note                                                 3
(Note                                                        3
[Asr Noise                                                   3
**Key Events                                                 2
Domain                                                       1
Domain Category                                      

## 5 · Duplicates & deterministic splits

* **Exact duplicates** – same sequence of (speaker label, normalised text) under a different id → dropped,
  keeping the first occurrence (logged in `duplicates.csv`). Duplicates inflate pattern counts and leak
  between splits 
* **Near-duplicates** – MinHash-LSH on word 3-gram shingles with numbers masked (so calls that differ only in
  ids / amounts still match), Jaccard ≥ `NEAR_DUP_THRESHOLD`. Flagged, not dropped.
* **Template groups** – the corpus looks partly template-generated, and train/test overlap of near-identical
  items inflates measured performance (Elangovan et al., EACL 2021). Calls with Jaccard ≥ `TEMPLATE_JACCARD`
  are linked into template groups. A **giant-group guard** raises the threshold until no group holds more than
  `TEMPLATE_MAX_GROUP_SHARE` of the calls, so chaining cannot swallow the corpus. Near-duplicate groups are
  always nested inside template groups.
* **Splits** – `train/dev/test` = 80/10/10 by hashing the **template group** id: deterministic across machines,
  library versions and row order, and a template never straddles two splits. The split is **locked here**,
  before any labelling, discovery, tuning or evaluation. Use *train* to discover hypotheses and fit models,
  *dev* to tune, and *test* only for the final numbers: the split-sample workflow of Egami et al.
  (*Science Advances* 2022). The template group is also the cluster unit for cluster-robust inference later.

In [6]:
def content_signature(rec: dict) -> str:
    payload = "\n".join(f"{t['label_norm']}|{t['text']}" for t in rec["turns"])
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()


dup_rows, sig_first = [], {}
for rec in records:
    if not rec["turns"]:
        rec["duplicate_of"] = None
        continue
    sig = content_signature(rec)
    rec["content_sha1"] = sig
    rec["duplicate_of"] = sig_first.get(sig)
    if rec["duplicate_of"] is None:
        sig_first[sig] = rec["transcript_id"]
    else:
        dup_rows.append({"transcript_id": rec["transcript_id"], "duplicate_of": rec["duplicate_of"], "type": "exact"})
n_exact = len(dup_rows)
n_turns_in_dropped = 0
if DROP_EXACT_DUPLICATES and n_exact:
    for rec in records:
        if rec["duplicate_of"] is not None:
            qc(rec["transcript_id"], "dedup", "exact_duplicate_dropped", f"duplicate of {rec['duplicate_of']}")
            n_turns_in_dropped += len(rec["turns"])
    records = [r for r in records if r["duplicate_of"] is None]
    kept_ids = {r["transcript_id"] for r in records}
    quar_df = quar_df[quar_df["transcript_id"].isin(kept_ids)].reset_index(drop=True)
print(f"exact duplicate conversations: {n_exact} ({'dropped' if DROP_EXACT_DUPLICATES else 'kept'}, "
      f"{n_turns_in_dropped} turns)")


def masked_words(rec: dict) -> list[str]:
    out = []
    for t in rec["turns"]:
        for w in t["text_clean"].split():
            out.append("<num>" if (w in NUMBER_WORDS or any(ch.isdigit() for ch in w)) else w)
    return out

MINHASH_PERM = 128
_MERSENNE = np.uint64((1 << 61) - 1)
_MAX32 = np.uint64((1 << 32) - 1)
_perm_rng = np.random.RandomState(SEED)
_PERM_A = _perm_rng.randint(1, (1 << 61) - 1, MINHASH_PERM, dtype=np.uint64)
_PERM_B = _perm_rng.randint(0, (1 << 61) - 1, MINHASH_PERM, dtype=np.uint64)


def shingle_hashes(words: list[str]) -> np.ndarray:
    grams = {" ".join(words[k:k + 3]) for k in range(len(words) - 2)}
    return np.array(sorted(int.from_bytes(hashlib.sha1(g.encode("utf-8")).digest()[:4], "little") for g in grams),
                    dtype=np.uint64)


def minhash_signature(hv: np.ndarray) -> np.ndarray:
    with np.errstate(over="ignore"):
        phv = ((hv[:, None] * _PERM_A[None, :] + _PERM_B[None, :]) % _MERSENNE) & _MAX32
    return phv.min(axis=0).astype(np.uint32)


LSH_BUCKET_CAP = 200        
LSH_ANCHORS = 50


def lsh_candidate_pairs(sig: np.ndarray, rows: int) -> np.ndarray:
    n = sig.shape[0]
    out = []
    for b in range(sig.shape[1] // rows):
        block = np.ascontiguousarray(sig[:, b * rows:(b + 1) * rows])
        keys = block.view(np.dtype((np.void, block.dtype.itemsize * rows))).ravel()
        order = np.argsort(keys, kind="stable")
        sk = keys[order]
        cuts = np.flatnonzero(sk[1:] != sk[:-1]) + 1
        for lo, hi in zip(np.r_[0, cuts], np.r_[cuts, n]):
            size = hi - lo
            if size < 2:
                continue
            m = np.sort(order[lo:hi]).astype(np.int64)
            if size <= LSH_BUCKET_CAP:
                ii, jj = np.triu_indices(size, 1)
            else:
                ii = np.concatenate([np.repeat(np.arange(LSH_ANCHORS), size), np.arange(size - 1)])
                jj = np.concatenate([np.tile(np.arange(size), LSH_ANCHORS), np.arange(1, size)])
                keep = ii < jj
                ii, jj = ii[keep], jj[keep]
            out.append(m[ii] * n + m[jj])
    return np.unique(np.concatenate(out)) if out else np.zeros(0, dtype=np.int64)


class SimilarityGraph:
    def __init__(self, hashes: list, sig: np.ndarray, min_threshold: float):
        import scipy.sparse as sp
        self.n = n = len(hashes)
        rows = 4 if min_threshold >= 0.45 else 3 if min_threshold >= 0.3 else 2   # P(found | J=0.5) = 0.87 with 4
        cand = lsh_candidate_pairs(sig, rows) if n > 1 else np.zeros(0, dtype=np.int64)
        i, j = cand // max(n, 1), cand % max(n, 1)
        est = np.empty(cand.size)
        for s in range(0, cand.size, 1_000_000):
            est[s:s + 1_000_000] = (sig[i[s:s + 1_000_000]] == sig[j[s:s + 1_000_000]]).mean(axis=1)
        keep = est >= min_threshold - 0.15          # 3.4 MinHash standard errors below the lowest threshold
        order = np.lexsort((cand[keep], -est[keep]))  # descending estimate, ties by pair index (deterministic)
        self.i, self.j, self.est = i[keep][order], j[keep][order], est[keep][order]
        self.jac = np.full(self.i.size, np.nan)
        self.sizes = np.array([h.size for h in hashes], dtype=np.int64)
        if n:
            vocab, inv = np.unique(np.concatenate(hashes), return_inverse=True)
            indptr = np.r_[0, np.cumsum(self.sizes)]
            self.M = sp.csr_matrix((np.ones(inv.size, dtype=np.float32), inv.ravel(), indptr), shape=(n, vocab.size))
        self.n_candidates = int(cand.size)

    def exact(self, k: np.ndarray) -> np.ndarray:
        todo = k[np.isnan(self.jac[k])]
        for s in range(0, todo.size, 20_000):
            t = todo[s:s + 20_000]
            a, b = self.i[t], self.j[t]
            inter = np.asarray(self.M[a].multiply(self.M[b]).sum(axis=1)).ravel()
            self.jac[t] = inter / (self.sizes[a] + self.sizes[b] - inter)
        return self.jac[k]

    def components(self, threshold: float) -> np.ndarray:
        from scipy.sparse import coo_matrix
        from scipy.sparse.csgraph import connected_components
        labels = np.arange(self.n)
        live = np.flatnonzero(self.est >= threshold - 0.15)
        start, size = 0, 2_000                 # blocks grow so that groups form before most pairs are looked at
        while start < live.size:
            k = live[start:start + size]
            start, size = start + size, min(2 * size, 200_000)
            k = k[labels[self.i[k]] != labels[self.j[k]]]      # already connected: cannot change the components
            if k.size:
                k = k[self.exact(k) >= threshold]
            if k.size:
                g = coo_matrix((np.ones(k.size), (labels[self.i[k]], labels[self.j[k]])), shape=(self.n, self.n))
                labels = connected_components(g, directed=False)[1][labels]
        return labels


all_ids = [r["transcript_id"] for r in records]
near_group = {tid: tid for tid in all_ids}
template_group = {tid: tid for tid in all_ids}
TEMPLATE_THRESHOLD_USED = None
if NEAR_DUP_DETECTION:
    t0 = time.perf_counter()
    ids_with, hashes = [], []
    for rec in records:
        hv = shingle_hashes(masked_words(rec))
        if hv.size:
            ids_with.append(rec["transcript_id"])
            hashes.append(hv)
    sig = np.stack([minhash_signature(h) for h in hashes]) if hashes else np.zeros((0, MINHASH_PERM), np.uint32)
    graph = SimilarityGraph(hashes, sig, min(TEMPLATE_JACCARD, NEAR_DUP_THRESHOLD))

    def groups_at(threshold: float) -> dict:
        labels = graph.components(threshold).tolist()
        rep = {}
        for tid, lab in zip(ids_with, labels):
            if lab not in rep or tid < rep[lab]:
                rep[lab] = tid
        out = {tid: tid for tid in all_ids}
        out.update({tid: rep[lab] for tid, lab in zip(ids_with, labels)})
        return out

    near_group = groups_at(NEAR_DUP_THRESHOLD)
    max_allowed = max(5, int(TEMPLATE_MAX_GROUP_SHARE * len(all_ids)))
    thr = TEMPLATE_JACCARD
    while True:
        tmpl = groups_at(thr)           
        largest = max(Counter(tmpl.values()).values())
        if largest <= max_allowed or thr >= NEAR_DUP_THRESHOLD - 1e-9:
            break
        print(f"  template threshold {thr:.2f}: largest group {largest} calls > {max_allowed} -> raising threshold")
        thr = round(min(thr + 0.1, NEAR_DUP_THRESHOLD), 2)
    template_group, TEMPLATE_THRESHOLD_USED = tmpl, thr
    print(f"similarity grouping done in {time.perf_counter() - t0:.1f}s ({graph.n_candidates:,} candidate pairs, "
          f"{int(np.isfinite(graph.jac).sum()):,} checked exactly; template threshold used: {thr:.2f})")
near_size, tmpl_size = Counter(near_group.values()), Counter(template_group.values())
n_near = sum(1 for tid in all_ids if near_size[near_group[tid]] > 1)
n_tmpl = sum(1 for tid in all_ids if tmpl_size[template_group[tid]] > 1)
print(f"near-duplicate clusters: {sum(1 for c in near_size.values() if c > 1)} covering {n_near} calls | "
      f"template groups with >1 call: {sum(1 for c in tmpl_size.values() if c > 1)} covering {n_tmpl} calls "
      f"(largest {max(tmpl_size.values()) if tmpl_size else 0})")
for tid, g in near_group.items():
    if near_size[g] > 1 and tid != g:
        dup_rows.append({"transcript_id": tid, "duplicate_of": g, "type": "near"})


def hash_split(key: str) -> str:
    u = int(hashlib.sha1(f"{SPLIT_SALT}:{key}".encode("utf-8")).hexdigest()[:12], 16) / float(16 ** 12)
    a, b, _ = SPLIT_FRACTIONS
    return "train" if u < a else ("dev" if u < a + b else "test")


for rec in records:
    tid = rec["transcript_id"]
    rec["dedup_group"] = near_group[tid]
    rec["near_dup_cluster_size"] = near_size[near_group[tid]]
    rec["template_group"] = template_group[tid]
    rec["template_group_size"] = tmpl_size[template_group[tid]]
    rec["split"] = hash_split(rec["template_group"])      # the split is locked here, by template group
dups_df = pd.DataFrame(dup_rows, columns=["transcript_id", "duplicate_of", "type"])
REPORT["duplicates"] = {"exact": n_exact, "exact_dropped": bool(DROP_EXACT_DUPLICATES and n_exact),
                        "turns_in_dropped_duplicates": n_turns_in_dropped,
                        "near_dup_conversations": int((dups_df["type"] == "near").sum()),
                        "template_threshold_used": TEMPLATE_THRESHOLD_USED,
                        "calls_in_multi_call_template_groups": int(n_tmpl),
                        "largest_template_group": int(max(tmpl_size.values())) if tmpl_size else 0}
print("split sizes:", dict(Counter(r["split"] for r in records)))

exact duplicate conversations: 0 (dropped, 0 turns)
similarity grouping done in 23.3s (191 candidate pairs, 0 checked exactly; template threshold used: 0.50)
near-duplicate clusters: 0 covering 0 calls | template groups with >1 call: 0 covering 0 calls (largest 1)
split sizes: {'train': 15663, 'test': 1957, 'dev': 2001}


## 6 · Speaker-role resolution

In [7]:
ROLE_CLIP = 6.0       
STRONG_SCORE = 8.0  
SUP_CTX_RE = re.compile(r"\b(?:supervisor|manager|team lead(?:er)?|escalat\w*|senior (?:agent|representative|advisor|colleague))\b")
SPEC_CTX_RE = re.compile(r"\b(?:specialist|expert|technician|underwriter|adjuster|investigator|department|"
                         r"(?:fraud|claims|billing|technical|loans?|retention|security) team)\b")
AGENT_SIDE = {"agent", "supervisor", "specialist", "system"}


def speaker_key(turn: dict) -> str:
    return turn["label_norm"] or "<empty>"


def conv_keys(rec: dict) -> list[str]:
    keys = [speaker_key(t) for t in rec["turns"]]
    unresolved = {k for k, t in zip(keys, rec["turns"]) if t["rule_role"] is None and PERSON_LABEL_RE.match(k)}
    alias = {}
    for k in unresolved:
        longer = [o for o in unresolved if o != k and o.startswith(k + " ")]
        if len(longer) == 1:
            alias[k] = longer[0]
    rec["key_alias"] = alias
    return [alias.get(k, k) for k in keys]


gold_mask_by_tid = {rec["transcript_id"]: {t["rule_role"] for t in rec["turns"]} == {"agent", "customer"} for rec in records}

rng = np.random.default_rng(SEED)
train_turns = [(t["text_clean"], 1 if t["rule_role"] == "agent" else 0)
               for rec in records if gold_mask_by_tid[rec["transcript_id"]] and rec["split"] == "train"
               for t in rec["turns"]]
if len(train_turns) > ROLE_CLF_MAX_TRAIN_TURNS:
    idx = rng.choice(len(train_turns), ROLE_CLF_MAX_TRAIN_TURNS, replace=False)
    train_turns = [train_turns[k] for k in idx]
print(f"role-classifier training turns: {len(train_turns):,}")
t0 = time.perf_counter()
role_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=300_000, sublinear_tf=True, dtype=np.float32)
X_train = role_vec.fit_transform([x for x, _ in train_turns])
y_train = np.array([y for _, y in train_turns])
role_clf = LogisticRegression(C=4.0, max_iter=2000, solver="liblinear", random_state=SEED)
role_clf.fit(X_train, y_train)
del X_train, y_train, train_turns
gc.collect()
print(f"trained in {time.perf_counter() - t0:.1f}s")

first_agent = [rec["turns"][0]["rule_role"] == "agent" for rec in records
               if gold_mask_by_tid[rec["transcript_id"]] and rec["split"] == "train"]
p_first = (sum(first_agent) + 1) / (len(first_agent) + 2)
PRIOR_FIRST = float(np.clip(math.log(p_first / (1 - p_first)), -4, 4))
print(f"P(agent speaks first) = {p_first:.4f} -> prior log-odds {PRIOR_FIRST:+.2f}")


def turn_logodds(texts: list[str]) -> np.ndarray:
    if not texts:
        return np.zeros(0)
    return role_clf.decision_function(role_vec.transform(texts))


def resolve_roles(turns: list[dict], keys: list[str], rule_roles: list, logodds) -> dict:
    order = list(dict.fromkeys(keys))
    rule = {k: r for k, r in zip(keys, rule_roles) if r is not None}
    result = {k: (r, "label_rule", None, None) for k, r in rule.items()}
    unresolved = [k for k in order if k not in rule]
    if not unresolved:
        return result
    scores = {k: 0.0 for k in unresolved}
    for k, lo in zip(keys, logodds):
        if k in scores:
            scores[k] += float(np.clip(lo, -ROLE_CLIP, ROLE_CLIP))
    if keys[0] in scores:
        scores[keys[0]] += PRIOR_FIRST
    has_customer = "customer" in rule.values()
    has_agent = "agent" in rule.values()
    assigned = {}
    if not has_customer:
        if len(unresolved) == 1:
            k = unresolved[0]
            if has_agent:   
                assigned[k] = ("customer" if scores[k] < STRONG_SCORE else "agent", "classifier_exclusion", scores[k], abs(scores[k]))
            else:
                assigned[k] = ("agent" if scores[k] > 0 else "customer", "classifier_sign", scores[k], abs(scores[k]))
        else:
            ranked = sorted(unresolved, key=lambda k: scores[k])
            assigned[ranked[0]] = ("customer", "classifier_rank", scores[ranked[0]], scores[ranked[1]] - scores[ranked[0]])
    cust_score = min((v[2] for v in assigned.values() if v[0] == "customer"), default=None)
    primary_agent = has_agent or any(v[0] == "agent" for v in assigned.values())
    for k in order:
        if k not in scores or k in assigned:
            continue
        if has_customer and scores[k] < -STRONG_SCORE:
            assigned[k] = ("customer", "classifier_strong", scores[k], abs(scores[k]))
            continue
        margin = scores[k] - cust_score if cust_score is not None else abs(scores[k])
        if not primary_agent:
            assigned[k] = ("agent", "classifier_rank", scores[k], margin)
            primary_agent = True
            continue
        first = keys.index(k)   # third-party company speaker: role from the context of their first turn
        ctx = " ".join(turns[j]["text_clean"] for j in range(max(0, first - 3), min(len(turns), first + 2)))
        role = "supervisor" if SUP_CTX_RE.search(ctx) else ("specialist" if SPEC_CTX_RE.search(ctx) else "agent")
        assigned[k] = (role, "context_rule", scores[k], margin)
    result.update(assigned)
    return result

val_recs = [r for r in records if r["split"] == "test" and gold_mask_by_tid[r["transcript_id"]]][:3000]
sup_recs = [r for r in records if r["split"] == "test"
            and {t["rule_role"] for t in r["turns"]} == {"agent", "customer", "supervisor"}][:1000]
val_lo = turn_logodds([t["text_clean"] for r in val_recs + sup_recs for t in r["turns"]])
turn_true = np.array([1 if t["rule_role"] == "agent" else 0 for r in val_recs for t in r["turns"]])
turn_acc = float(((val_lo[:len(turn_true)] > 0).astype(int) == turn_true).mean()) if len(turn_true) else float("nan")


def hidden_label_eval(recs, lo_all, offset, hide):
    ok_spk = n_spk = ok_conv = 0
    pos = offset
    for r in recs:
        n = len(r["turns"])
        lo = lo_all[pos:pos + n]
        pos += n
        keys = [f"spk_{t['rule_role']}" if t["rule_role"] in hide else speaker_key(t) for t in r["turns"]]
        rules = [None if t["rule_role"] in hide else t["rule_role"] for t in r["turns"]]
        res = resolve_roles(r["turns"], keys, rules, lo)
        oks = [res[f"spk_{role}"][0] == role for role in hide]
        ok_spk, n_spk, ok_conv = ok_spk + sum(oks), n_spk + len(oks), ok_conv + all(oks)
    return ok_spk / max(n_spk, 1), ok_conv / max(len(recs), 1)


spk2, conv2 = hidden_label_eval(val_recs, val_lo, 0, ("agent", "customer"))
n_val_turns = sum(len(r["turns"]) for r in val_recs)
spk3a, conv3a = hidden_label_eval(sup_recs, val_lo, n_val_turns, ("agent", "customer"))
spk3b, conv3b = hidden_label_eval(sup_recs, val_lo, n_val_turns, ("agent", "customer", "supervisor"))
ROLE_EVAL = {"turn_level_accuracy": round(turn_acc, 4),
             "two_party_speaker_accuracy": round(spk2, 4), "two_party_conversation_accuracy": round(conv2, 4),
             "with_labelled_supervisor_conversation_accuracy": round(conv3a, 4),
             "three_party_all_hidden_conversation_accuracy": round(conv3b, 4),
             "n_two_party_validation_calls": len(val_recs), "n_supervisor_validation_calls": len(sup_recs),
             "prior_first_logodds": round(PRIOR_FIRST, 3)}
REPORT["role_resolution_validation"] = ROLE_EVAL
print(json.dumps(ROLE_EVAL, indent=2))
if len(val_recs) and ROLE_EVAL["two_party_conversation_accuracy"] < 0.97:
    print("WARNING: role resolution recovers < 97% of held-out calls; inspect speaker_role_map.csv")

role-classifier training turns: 200,000
trained in 7.3s
P(agent speaks first) = 0.9999 -> prior log-odds +4.00
{
  "turn_level_accuracy": 0.9763,
  "two_party_speaker_accuracy": 0.9995,
  "two_party_conversation_accuracy": 0.9995,
  "with_labelled_supervisor_conversation_accuracy": 1.0,
  "three_party_all_hidden_conversation_accuracy": 0.9167,
  "n_two_party_validation_calls": 1852,
  "n_supervisor_validation_calls": 12,
  "prior_first_logodds": 4.0
}


### 6.1 Apply to every conversation

In [8]:
t0 = time.perf_counter()
KEYS = {rec["transcript_id"]: conv_keys(rec) for rec in records}
need = [(ri, ti) for ri, rec in enumerate(records) for ti, t in enumerate(rec["turns"]) if t["rule_role"] is None]
need_lo = turn_logodds([records[ri]["turns"][ti]["text_clean"] for ri, ti in need])
lo_map = {(ri, ti): lo for (ri, ti), lo in zip(need, need_lo)}
ROLE_ROWS = []
for ri, rec in enumerate(records):
    turns = rec["turns"]
    rec["speaker_map"] = {}
    if not turns:
        continue
    keys = KEYS[rec["transcript_id"]]
    res = resolve_roles(turns, keys, [t["rule_role"] for t in turns], [lo_map.get((ri, ti), 0.0) for ti in range(len(turns))])
    for k in dict.fromkeys(keys):
        role, method, score, margin = res[k]
        label = next(t["label_clean"] for t in turns if speaker_key(t) == k)   # aliases map onto an existing label
        is_person = method != "label_rule" and bool(PERSON_LABEL_RE.match(k))
        low_conf = method.startswith("classifier") and (margin is None or margin < ROLE_MIN_MARGIN)
        rec["speaker_map"][k] = {"role": role, "method": method, "label": label,
                                 "speaker_name": label if is_person else None}
        ROLE_ROWS.append({"transcript_id": rec["transcript_id"], "label": label, "label_norm": k,
                          "aliases": ",".join(a for a, b in rec["key_alias"].items() if b == k),
                          "role": role, "method": method,
                          "score": None if score is None else round(score, 3),
                          "margin": None if margin is None else round(margin, 3), "low_confidence": low_conf,
                          "n_turns": keys.count(k)})
role_map_df = pd.DataFrame(ROLE_ROWS)
print(f"resolved in {time.perf_counter() - t0:.1f}s")
display(role_map_df.groupby(["method", "role"]).size().rename("speakers").reset_index())
print("low-confidence speaker assignments:", int(role_map_df["low_confidence"].sum()))
print("name aliases merged:", int((role_map_df["aliases"] != "").sum()))
REPORT["role_methods"] = role_map_df.groupby(["method", "role"]).size().rename("n").reset_index().to_dict("records")

resolved in 1.4s


,method,role,speakers
0,classifier_rank,agent,1126
1,classifier_rank,customer,1126
2,context_rule,agent,2
3,context_rule,supervisor,3
4,label_rule,agent,18495
5,label_rule,customer,18495
6,label_rule,specialist,4
7,label_rule,supervisor,162
8,label_rule,system,4


low-confidence speaker assignments: 0
name aliases merged: 34


## 7 · Assemble the turn table

In [9]:
ROLE_DISPLAY = {"agent": "Agent", "customer": "Customer", "supervisor": "Supervisor", "specialist": "Specialist", "system": "System"}
rows = []
for rec in records:
    turns, keys = rec["turns"], KEYS[rec["transcript_id"]]
    n = len(turns)
    prev_key, block = None, -1
    for ti, (t, k) in enumerate(zip(turns, keys)):
        sm = rec["speaker_map"][k]
        if k != prev_key:
            block += 1
        rows.append({
            "transcript_id": rec["transcript_id"], "turn_id": ti, "turn_uid": f"{rec['transcript_id']}:{ti}",
            "orig_index": t["orig_index"], "sub_index": t["sub_index"], "from_embedded": t["from_embedded"],
            "speaker_label_raw": t["label_raw"], "speaker": ROLE_DISPLAY[sm["role"]],
            "party": "Customer" if sm["role"] == "customer" else "Agent", "speaker_name": sm["speaker_name"],
            "role_method": sm["method"], "block_id": block, "same_speaker_as_prev": k == prev_key,
            "rel_position": round(ti / (n - 1), 4) if n > 1 else 0.0,
            "text_raw": t["text_raw"], "text": t["text"], "text_clean": t["text_clean"],
            "nonverbal_types": [e["type"] for e in t["nonverbal"]], "nonverbal_raw": [e["raw"] for e in t["nonverbal"]],
        })
        prev_key = k
turns_df = pd.DataFrame(rows)
turns_df["phase"] = np.select([turns_df["rel_position"] <= 0.2, turns_df["rel_position"] >= 0.8],
                              ["opening", "closing"], default="middle")
turns_df["party_turn_index"] = turns_df.groupby(["transcript_id", "party"]).cumcount()
print(f"turns: {len(turns_df):,} | conversations with turns: {turns_df['transcript_id'].nunique():,}")
print(turns_df["speaker"].value_counts().to_string())

turns: 686,196 | conversations with turns: 19,621
speaker
Agent         349792
Customer      335387
Supervisor       996
Specialist        16
System             5


## 8 · Metadata normalisation, brand & business domain

In [10]:
def slug(s) -> str:
    s = normalize_text(str(s)).replace("&", " and ")
    return re.sub(r"[^a-z0-9]+", "_", s).strip("_") or "unknown"


BRAND_RE = re.compile(
    r"\b(?:thank(?:s| you)(?: so much)? for (?:calling|contacting|choosing|reaching out to|reaching)|welcome to|"
    r"you(?:'ve| have) reached)\s+(?:the\s+)?(?P<brand>[a-z0-9&'\-.]+(?:\s+[a-z0-9&'\-.]+){0,3}?)"
    r"(?=\s+(?:this is|my name|how|you're|you are|i'm|i am|speaking|where|customer|technical|tech|support|service|"
    r"services|department|desk|team|and|we|what|can|may|good|today|hello|hi)\b|\s*$)")

BRAND_NOISE_RE = re.compile(r"\b(?:you know|i mean|like|so|basically|actually|um|uh)\b")

DOMAIN_LEXICON = {
    "hospitality": [(r"\bhotels?\b", 3), (r"\bresorts?\b", 3), (r"\bsuites?\b", 2), (r"\binns?\b", 2),
                    (r"\blodg(?:e|es|ing)\b", 3), (r"\bmotels?\b", 3), (r"\bhousekeeping\b", 3), (r"\bfront desk\b", 3),
                    (r"\bconcierge\b", 3), (r"\bamenit(?:y|ies)\b", 2), (r"\bcheck[- ]?(?:in|out)\b", 1),
                    (r"\brooms?\b", 1), (r"\b(?:per|a|one|two|three|four|five|six|seven) nights?\b", 2),
                    (r"\bguests?\b", 1), (r"\b(?:room service|minibar|king bed|queen bed|double bed)\b", 3)],
    "airline": [(r"\bflights?\b", 3), (r"\bairlines?\b", 3), (r"\bairports?\b", 3), (r"\bairways\b", 3),
                (r"\bboarding(?: pass)?\b", 2), (r"\bpnr\b", 3), (r"\blayovers?\b", 3), (r"\brebook\w*\b", 2),
                (r"\bfares?\b", 2), (r"\b(?:baggage|luggage)\b", 2), (r"\bfrequent flyer\b", 3), (r"\bmiles\b", 1),
                (r"\bdepart\w*\b", 1), (r"\bgates?\b", 1), (r"\bitinerar(?:y|ies)\b", 2),
                (r"\b(?:economy|business|first) class\b", 2), (r"\b(?:pilot|crew|aircraft|runway|terminal)\b", 2),
                (r"\btsa\b", 2), (r"\bjets?\b", 1)],
    "telecom": [(r"\binternet\b", 3), (r"\bwi-?fi\b", 3), (r"\brouters?\b", 3), (r"\bmodems?\b", 3),
                (r"\bbroadband\b", 3), (r"\boutages?\b", 2), (r"\bnetwork\b", 2), (r"\bconnectivity\b", 2),
                (r"\bconnection\b", 1), (r"\bsignal\b", 2), (r"\b(?:mbps|megabits?|gigabits?|bandwidth)\b", 3),
                (r"\b(?:data|mobile|phone|cell|wireless) plans?\b", 3), (r"\bunlimited data\b", 3),
                (r"\bdropped calls?\b", 3), (r"\bsim(?: card)?\b", 2), (r"\b(?:5g|4g|lte)\b", 3),
                (r"\bfib(?:er|re)\b", 2), (r"\bcarriers?\b", 1), (r"\bstreaming\b", 1), (r"\btechnician\b", 1),
                (r"\bdata usage\b", 3), (r"\bhotspot\b", 3)],
    "banking": [(r"\bloans?\b", 3), (r"\bcredit cards?\b", 3), (r"\bcredit limit\b", 3), (r"\bdebit cards?\b", 3),
                (r"\bbank(?:ing)?\b", 3), (r"\bapr\b", 3), (r"\binterest rates?\b", 2), (r"\bmortgages?\b", 3),
                (r"\boverdraft\b", 3), (r"\b(?:checking|savings) accounts?\b", 3), (r"\btransactions?\b", 2),
                (r"\bfraud\w*\b", 1), (r"\batms?\b", 3), (r"\bstatements?\b", 1),
                (r"\b(?:annual|late|maintenance|monthly|account|foreign transaction) fees?\b", 2),
                (r"\bcredit score\b", 3), (r"\b(?:wire|balance) transfer\b", 3), (r"\bcardholder\b", 2),
                (r"\bunauthori[sz]ed\b", 1)],
    "insurance": [(r"\binsur(?:ance|er|ers|ed)\b", 3), (r"\bpolicy ?holders?\b", 3), (r"\bpremiums?\b", 1),
                  (r"\bdeductibles?\b", 3), (r"\bclaims?\b", 2), (r"\bcoverage\b", 1), (r"\bunderwrit\w*\b", 3),
                  (r"\bbeneficiar(?:y|ies)\b", 3), (r"\b(?:life|health|auto|car|home|travel|pet|renters?) insurance\b", 3),
                  (r"\bco-?pays?\b", 3), (r"\b(?:my|your) policy\b", 1), (r"\bpolicy (?:number|renewal|documents?|terms)\b", 2),
                  (r"\badjusters?\b", 3), (r"\bout of pocket\b", 2), (r"\b(?:in|out of)[- ]network\b", 2), (r"\bpayout\b", 2)],
    "retail_ecommerce": [(r"\b(?:my|the|this|your|an|online) orders?\b", 3), (r"\border (?:number|id|status|confirmation)\b", 3),
                         (r"\bdeliver(?:y|ies|ed)\b", 2), (r"\bpackages?\b", 2), (r"\bparcels?\b", 3),
                         (r"\bship(?:ping|ped|ment|ments)\b", 3), (r"\btracking(?: number| info)?\b", 2),
                         (r"\bcouriers?\b", 3), (r"\bwarehouse\b", 3), (r"\bproducts?\b", 1), (r"\bitems?\b", 1),
                         (r"\bdefective\b", 2), (r"\bdamaged\b", 1), (r"\breturn (?:label|policy|window|request)\b", 2),
                         (r"\breplacements?\b", 1), (r"\bwarranty\b", 2), (r"\b(?:cart|checkout)\b", 1),
                         (r"\b(?:out of|in) stock\b", 2)],
}
DOMAIN_LEXICON = {d: [(re.compile(p), w) for p, w in pats] for d, pats in DOMAIN_LEXICON.items()}


def score_domains(text: str) -> dict:
    return {d: sum(w * min(len(rx.findall(text)), 3) for rx, w in pats) for d, pats in DOMAIN_LEXICON.items()}


def vote_domain(scores: dict) -> tuple[str, float]:
    ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    (d1, s1), (_, s2) = ranked[0], ranked[1]
    if s1 < 3 or s1 == s2:
        return "unknown", 0.0
    return d1, round((s1 - s2) / s1, 3)


def parse_time(value):
    if value is None or (isinstance(value, float) and math.isnan(value)) or not str(value).strip():
        return pd.NaT
    ts = pd.to_datetime(str(value).strip(), format="%Y-%m-%d %H:%M:%S", errors="coerce")
    if pd.isna(ts):
        ts = pd.to_datetime(str(value).strip(), errors="coerce")
    return ts


conv_rows = []
for rec in records:
    tid = rec["transcript_id"]
    open_turns = rec["turns"][:OPENING_TURNS]
    intent = rec.get("intent")
    intent = WS_RE.sub(" ", str(intent)).strip() if intent is not None and str(intent).strip() else None
    reason = rec.get("reason_for_call")
    reason = WS_RE.sub(" ", unicodedata.normalize("NFKC", str(reason)).translate(QUOTE_MAP)).strip() \
        if reason is not None and str(reason).strip() else None
    ts = parse_time(rec.get("time_of_interaction"))
    if pd.isna(ts) and rec.get("time_of_interaction") not in (None, ""):
        qc(tid, "metadata", "time_parse_failed", rec.get("time_of_interaction"))
    for k in ("time_of_interaction", "intent", "reason_for_call", "domain"):
        if rec.get(k) is None or not str(rec.get(k)).strip():
            qc(tid, "metadata", f"missing_{k}")
    agent_open = " ".join(t["text_clean"] for t, k in zip(open_turns, KEYS[tid][:OPENING_TURNS])
                          if rec["speaker_map"][k]["role"] in AGENT_SIDE)
    m = BRAND_RE.search(agent_open)
    brand = WS_RE.sub(" ", BRAND_NOISE_RE.sub(" ", m.group("brand"))).strip() if m else None
    brand = brand or None
    dom_text = " ".join([(intent or "").lower(), (reason or "").lower()] + [t["text_clean"] for t in rec["turns"][:6]])
    conv_dom, conv_dom_conf = vote_domain(score_domains(dom_text))
    open_text = WS_RE.sub(" ", BRAND_NOISE_RE.sub(" ", " ".join(t["text_clean"] for t in open_turns))).strip()
    open_dom, open_dom_conf = vote_domain(score_domains(open_text))
    conv_rows.append({
        "transcript_id": tid, "file_order": rec["file_order"],
        "time_of_interaction": ts.strftime("%Y-%m-%d %H:%M:%S") if not pd.isna(ts) else None,
        "call_date": ts.strftime("%Y-%m-%d") if not pd.isna(ts) else None,
        "call_hour": int(ts.hour) if not pd.isna(ts) else -1,
        "call_weekday": ts.day_name() if not pd.isna(ts) else None,
        "domain_raw": rec.get("domain"), "intent": intent, "intent_key": slug(intent) if intent else "unknown",
        "reason_for_call": reason, "brand": brand, "domain_keyword_vote": conv_dom,
        "domain_keyword_confidence": conv_dom_conf, "opening_domain_vote": open_dom,
        "opening_domain_confidence": open_dom_conf,
        **({"extra_meta_json": json.dumps(rec["extra_meta"], ensure_ascii=False, default=str)} if rec["extra_meta"] else {}),
    })
conv_df = pd.DataFrame(conv_rows)

split_of = {r["transcript_id"]: r["split"] for r in records}
vote_rows = conv_df[(conv_df["domain_keyword_vote"] != "unknown") & (conv_df["transcript_id"].map(split_of) == "train")]
votes = vote_rows.groupby(["intent_key", "domain_keyword_vote"]).size().unstack(fill_value=0)
dm_rows = []
for ik, row in votes.iterrows():
    tot = int(row.sum())
    ranked = row.sort_values(ascending=False)
    top, second = ranked.index[0], (ranked.index[1] if len(ranked) > 1 else None)
    purity = ranked.iloc[0] / tot if tot else 0.0
    if ik in DOMAIN_OVERRIDES:
        decision = "manual_override"
    elif purity >= DOMAIN_INTENT_PURITY and tot >= 10:
        decision = "intent_majority"
    else:
        decision = "per_conversation_vote"
    dm_rows.append({"intent_key": ik, "n_voted": tot, "majority_domain": DOMAIN_OVERRIDES.get(ik, top),
                    "purity": round(float(purity), 3), "second_domain": second,
                    "second_share": round(float(ranked.iloc[1] / tot), 3) if second is not None and tot else 0.0,
                    "decision": decision})
domain_map_df = pd.DataFrame(dm_rows, columns=["intent_key", "n_voted", "majority_domain", "purity", "second_domain", "second_share", "decision"])
intent_names = conv_df.groupby("intent_key")["intent"].agg(lambda s: s.dropna().mode().iloc[0] if s.notna().any() else None)
domain_map_df.insert(1, "intent", domain_map_df["intent_key"].map(intent_names))
dm_lookup = domain_map_df.set_index("intent_key")


def final_domain(row) -> tuple[str, str]:
    ik = row["intent_key"]
    if ik in DOMAIN_OVERRIDES:
        return DOMAIN_OVERRIDES[ik], "manual_override"
    if ik in dm_lookup.index and dm_lookup.loc[ik, "decision"] == "intent_majority":
        return dm_lookup.loc[ik, "majority_domain"], "intent_majority"
    if row["domain_keyword_vote"] != "unknown":
        return row["domain_keyword_vote"], "conversation_keywords"
    if ik in dm_lookup.index:
        return dm_lookup.loc[ik, "majority_domain"], "intent_majority_fallback"
    return "unknown", "none"


fd = conv_df.apply(final_domain, axis=1, result_type="expand")
conv_df["business_domain"], conv_df["business_domain_source"] = fd[0], fd[1]
print(domain_map_df.sort_values(["majority_domain", "intent_key"]).to_string(index=False))
print("\nbusiness_domain:", conv_df["business_domain"].value_counts().to_dict())
print("source:", conv_df["business_domain_source"].value_counts().to_dict())
print("keyword vote agrees with final domain:",
      f"{(conv_df['domain_keyword_vote'] == conv_df['business_domain']).mean():.1%}")
known_open = conv_df["opening_domain_vote"] != "unknown"
print(f"opening-only vote: decided for {known_open.mean():.1%} of calls; agrees with business_domain on "
      f"{(conv_df.loc[known_open, 'opening_domain_vote'] == conv_df.loc[known_open, 'business_domain']).mean():.1%} of those")
print("\ntop brands:", conv_df["brand"].value_counts().head(20).to_dict())
REPORT["business_domain_counts"] = conv_df["business_domain"].value_counts().to_dict()
REPORT["opening_domain_vote_counts"] = conv_df["opening_domain_vote"].value_counts().to_dict()

              intent_key                  intent  n_voted  majority_domain  purity    second_domain  second_share              decision
    cross_brand_mentions    Cross-Brand Mentions      411          airline   0.844        insurance         0.156       intent_majority
        delay_management        Delay Management      585          airline   1.000          banking         0.000       intent_majority
         loyalty_program         Loyalty Program      518          airline   0.471 retail_ecommerce         0.425 per_conversation_vote
       price_sensitivity       Price Sensitivity      418          airline   0.998        insurance         0.002       intent_majority
           refund_policy           Refund Policy      439          airline   1.000          banking         0.000       intent_majority
      urgency_and_stress        Urgency & Stress      421          airline   0.938        insurance         0.062       intent_majority
   credit_limit_requests   Credit Limit Requests

## 9 · Turn-level enrichment: conversational-dynamics signals

In [11]:
CUE_DEFS = {
    # agent-side moves
    "hold_request": ("Agent", r"\b(?:(?:one|a|just a|just one|a quick|a brief|give me a|give me one)\s+(?:moment|minute|min|second|sec)s?\b|"
                              r"please hold|hold on|bear with me|(?:let me|i'll|i will|i'm going to|i am going to|allow me to)\s+(?:just\s+|quickly\s+)?"
                              r"(?:check|look|pull|see|verify|review|investigate|find|access|bring up|go through)\b|"
                              r"while i (?:check|look|pull|review|verify|see)|(?:put|place) you on (?:a )?(?:brief |short |quick )?hold)"),
    "hold_return": ("Agent", r"\b(?:thank(?:s| you)(?: so much| very much)? for (?:holding|waiting|your patience|being patient|bearing with me|hanging on)|"
                             r"sorry (?:for|about) (?:the|that) (?:wait|delay|hold)|sorry (?:for|about) keeping you (?:on hold|waiting)|appreciate your patience)\b"),
    "apology": ("Any", r"\b(?:sorry|apologi[sz]e|apologi[sz]ing|apologies)\b"),
    "empathy": ("Agent", r"\b(?:i (?:completely |totally |really |fully |absolutely )?understand|i can (?:imagine|see how|see why|understand)|"
                         r"(?:that|this|it) (?:must be|sounds|is|can be) (?:really |so |very |incredibly |quite |a bit |pretty )?"
                         r"(?:frustrating|stressful|difficult|upsetting|annoying|inconvenient|concerning|hard|worrying|disappointing)|"
                         r"i hear you|i know how (?:frustrating|stressful|difficult|upsetting)|i(?:'m| am) (?:so |really |very )?sorry to hear)\b"),
    "refusal": ("Agent", r"\b(?:unfortunately|i(?:'m| am) afraid|(?:i|we) (?:can't|cannot|can not|won't be able|will not be able|am unable|are unable|"
                         r"am not able|are not able|don't have the ability|do not have the ability)|(?:not|isn't|is not|aren't|are not) "
                         r"(?:eligible|possible|covered|refundable|allowed|permitted)|(?:against|per|according to|due to|under) (?:our |the |company )?"
                         r"polic(?:y|ies)|non-?refundable|no exceptions?)\b"),
    "offer_solution": ("Agent", r"\b(?:what i can do|i can (?:offer|provide|give|issue|waive|credit|refund|upgrade|rebook|replace|send|arrange|schedule|"
                                r"apply|process|extend|reverse|expedite)|i(?:'ve| have| will|'ll) (?:gone ahead and )?(?:offered|issued|waived|credited|"
                                r"refunded|upgraded|rebooked|replaced|sent|arranged|scheduled|applied|processed|extended|reversed|expedited|added|flagged)|"
                                r"as a (?:small )?(?:courtesy|goodwill gesture)|we can offer)\b"),
    "agent_escalation": ("Agent", r"\b(?:escalat(?:e|ed|ing|ion)|(?:my|a|the|our) (?:supervisor|manager|team lead(?:er)?|senior (?:agent|representative|colleague)))\b"),
    "agent_transfer": ("Agent", r"\b(?:transfer(?:ring)? (?:you|your call|the call|this call)|(?:connect|put) you (?:through )?(?:to|with) (?:a|the|my|our|someone)|"
                                r"(?:another|the relevant|the appropriate|a different|a specialized|our) (?:department|team)|(?:a|our|the) specialist)\b"),
    # customer-side states and intents
    "escalation_request": ("Customer", r"\b(?:(?:speak|talk|chat)(?:ing)? (?:to|with) (?:(?:a|your|the) )?(?:supervisor|manager|superior|boss)|"
                                       r"(?:speak|talk) (?:to|with) (?:someone|somebody) (?:else|senior|higher(?: up)?|in charge|more senior|with (?:more )?authority)|"
                                       r"(?:get|want|need) (?:a|your|the) (?:supervisor|manager)|escalate (?:this|it|my (?:case|issue|complaint|claim|request|problem)))\b"),
    "churn_intent": ("Customer", r"\b(?:(?:cancel(?:l?ing|l?ed|s)?|clos(?:e|es|ed|ing)|terminat(?:e|es|ed|ing)|end(?:s|ed|ing)?|discontinu(?:e|es|ed|ing)|"
                                 r"drop(?:s|ped|ping)?) (?:my|the|our) (?:\w+ )?(?:account|policy|subscription|service|"
                                 r"membership|contract|coverage|plan)|switch(?:ing)? (?:over )?(?:to )?(?:a |an |another |some )?(?:different |new |other )?"
                                 r"(?:provider|carrier|company|bank|insurer|airline|hotel chain|brand|competitor)s?|(?:take|taking|move|moving) my business|"
                                 r"(?:leave|leaving|quit|quitting) (?:you guys|your (?:company|service|bank|airline|network))|"
                                 r"(?:not|won't|will not|don't want to|do not want to|not going to) renew|(?:go|going|went) with (?:a |another |the )?"
                                 r"(?:competitor|other (?:company|provider|carrier|airline|bank|insurer))|(?:considering|thinking (?:about|of)) "
                                 r"(?:switching|leaving|cancell?ing|other (?:providers|options|companies|carriers|airlines|banks|insurers)))\b"),
    "refund": ("Any", r"\b(?:refund(?:s|ed|ing|able)?|reimburs(?:e|ed|ement|ing)|money back|charge ?back|compensat(?:e|ed|ion)|"
                      r"(?:travel|account|store) credit|credit (?:it |that )?back)\b"),
    "negative_emotion": ("Customer", r"\b(?:frustrat\w*|annoy\w*|upset|angry|mad|furious|ridiculous|unacceptable|disappoint\w*|terrible|horrible|"
                                     r"awful|worst|fed up|sick (?:and tired )?of|tired of|waste of (?:my )?time|not happy|unhappy|stress(?:ed|ful)?|"
                                     r"outrag\w*|infuriat\w*|irritat\w*|livid|absurd|nightmare|pathetic|useless|incompetent|unbelievable|disgusted|appalled)\b"),
    "confusion": ("Customer", r"\b(?:i(?:'m| am) (?:(?:a bit|a little|kind of|so|really|very) )?(?:confused|lost)|not sure i (?:follow|understand)|"
                              r"i (?:don't|do not|didn't|did not) (?:understand|get it|follow|get what)|what do you mean|what does that mean|"
                              r"that(?:'s| is) not what i (?:said|asked|meant)|(?:can|could) you (?:please )?(?:repeat|explain|clarify|say that again|rephrase)|"
                              r"(?:didn't|did not) (?:catch|hear|understand|get) that|makes? no sense|(?:doesn't|does not) make (?:any )?sense|"
                              r"pardon|come again|you(?:'re| are) not (?:listening|understanding|hearing me))\b"),
    "repeat_contact": ("Customer", r"\b(?:i(?:'ve| have)? already (?:told|said|explained|gave|given|sent|provided|called|tried|done)|as i (?:already )?"
                                   r"(?:said|mentioned|explained|told you)|like i (?:said|told you|mentioned)|for the (?:second|third|fourth|fifth) time|"
                                   r"(?:called|calling|contacted|reached out)(?: you(?: guys)?)? (?:back |in )?(?:again|multiple times|several times|many times|"
                                   r"twice|three times|so many times|before)|(?:this is|it's) (?:my|the) (?:second|third|fourth|fifth) (?:call|time)|"
                                   r"(?:second|third|fourth|fifth) time i(?:'ve| have)? called|every time i call|keep (?:calling|having to call))\b"),
    "issue_persistence": ("Customer", r"\bstill (?:hasn't|has not|haven't|have not|isn't|is not|not|no|doesn't|does not|don't|do not|can't|cannot|waiting|nothing|broken|down|pending|under review)\b"),
    "competitor_mention": ("Customer", r"\b(?:competitors?|competition|other (?:providers?|carriers?|companies|airlines?|banks?|insurers?|insurance companies|"
                                       r"hotels?|brands?|stores?)|(?:another|a different) (?:provider|carrier|company|airline|bank|insurer|hotel|brand|store))\b"),
}
CUES = {name: (scope, re.compile(p)) for name, (scope, p) in CUE_DEFS.items()}
DM_STRIP_RE = re.compile(r"\b(?:you know|i mean|like)\b")


def cue_view(clean: str) -> str:
    return WS_RE.sub(" ", DM_STRIP_RE.sub(" ", clean)).strip()


LEADING_DM_RE = re.compile(r"^(?:(?:so|and|but|well|okay|ok|alright|all right|oh|hi|hello|hey|yeah|yes|no|right|sure|now|then|also|just|"
                           r"sorry|like|you know|i mean|great|perfect|thanks|thank you)\s+)+")
QUESTION_START_RE = re.compile(r"^(?:what|why|how|when|where|which|who|whom|whose|is|are|was|were|am|do|does|did|can|could|would|will|"
                               r"should|shall|may|might|have|has|isn't|aren't|wasn't|weren't|don't|doesn't|didn't|can't|couldn't|"
                               r"wouldn't|won't|shouldn't|haven't|hasn't)\b")
QUESTION_ANY_RE = re.compile(  # question frames that also occur mid-turn (text has no punctuation)
    r"\b(?:(?:can|could|would|will) you (?:please |just |also )?(?:tell|explain|check|confirm|help|send|let|give|show|walk|clarify|"
    r"repeat|look|provide|verify|spell|hold)|(?:how|what) (?:can|may) i (?:help|assist|do)|is there (?:anything|any|a way|something)|"
    r"would you like|do you (?:know|have|need|want|mind|remember)|(?:may|can|could) i (?:have|get|ask|confirm|put|place)|"
    r"is (?:that|this|it) (?:right|correct|okay|possible|true)|(?:when|why|how long|how much|how many) (?:is|was|are|were|will|would|"
    r"did|do|does|can|should)|i was wondering|any chance|does that (?:make sense|work|help|sound))\b")
BACKCHANNEL_RE = re.compile(r"^(?:(?:uh huh|mm hmm|mhm|mm|hmm|um|uh|yeah|yep|yes|yup|okay|ok|right|sure|alright|all right|i see|got it|"
                            r"gotcha|oh|ah|go on|go ahead|understood|mm-hmm)\s*)+$")


def is_question(clean: str, raw: str) -> bool:
    if "?" in raw:
        return True
    t = LEADING_DM_RE.sub("", clean)
    return bool(QUESTION_START_RE.match(t) or QUESTION_ANY_RE.search(clean))

assert CUES["escalation_request"][1].search("i want to speak to a supervisor right now")
assert CUES["escalation_request"][1].search(cue_view("that's unacceptable i want to speak to a you know supervisor i mean right now"))
assert CUES["repeat_contact"][1].search(cue_view("this is the third you know time i mean i've called about this"))
assert CUES["churn_intent"][1].search("i'm closing my account")
assert CUES["churn_intent"][1].search("i'm thinking about switching to another provider")
assert not CUES["churn_intent"][1].search("i need to cancel my booking for friday")
assert CUES["hold_request"][1].search("let me check that for you one moment please")
assert CUES["refusal"][1].search("unfortunately per our policy i'm not able to change that")
assert CUES["repeat_contact"][1].search("this is the third time i've called about this")
assert is_question("um can you check my balance", "um can you check my balance") and not is_question("i can check that", "i can check that")
assert is_question("thank you for calling x this is sam how can i help you today", "")
assert BACKCHANNEL_RE.match("uh huh") and not BACKCHANNEL_RE.match("okay so my card was charged twice")

t0 = time.perf_counter()
texts, cleans, parties = turns_df["text"].tolist(), turns_df["text_clean"].tolist(), turns_df["party"].tolist()
cue_texts = [cue_view(c) for c in cleans]
turns_df["n_words"] = [len(WORD_RE.findall(t)) for t in texts]
turns_df["n_chars"] = [len(t) for t in texts]
turns_df["n_filled_pauses"] = [count_filled_pauses(t) for t in texts]
turns_df["n_repetitions"] = [count_repetitions(t) for t in texts]
turns_df["n_hedges"] = [len(HEDGE_RE.findall(c)) for c in cleans]
turns_df["n_discourse_markers"] = [len(DISCOURSE_MARKER_RE.findall(c)) for c in cleans]
turns_df["filled_pause_rate"] = (turns_df["n_filled_pauses"] / turns_df["n_words"].clip(lower=1)).round(4)
turns_df["is_backchannel"] = [bool(BACKCHANNEL_RE.match(c)) and len(c.split()) <= 4 for c in cleans]
turns_df["is_question"] = [is_question(c, t) for c, t in zip(cleans, texts)]
for name, (scope, rx) in CUES.items():
    turns_df[f"cue_{name}"] = [bool(rx.search(c)) if scope in ("Any", p) else False for c, p in zip(cue_texts, parties)]
CUE_COLS = [f"cue_{n}" for n in CUES]
cue_matrix = turns_df[CUE_COLS].to_numpy()
turns_df["cues"] = [[CUE_COLS[j][4:] for j in np.flatnonzero(row)] for row in cue_matrix]
print(f"lexical features in {time.perf_counter() - t0:.1f}s")

t0 = time.perf_counter()
rep_vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2, sublinear_tf=True, dtype=np.float32)
fit_mask = turns_df["transcript_id"].map(split_of).eq("train").to_numpy()   # vocabulary / idf from train calls only
rep_vec.fit([c for c, m in zip(cue_texts, fit_mask) if m] or cue_texts)
X_all = rep_vec.transform(cue_texts)
is_unigram = np.array([" " not in f for f in rep_vec.get_feature_names_out()])
row_of_nnz = np.repeat(np.arange(X_all.shape[0]), np.diff(X_all.indptr))
n_content = np.bincount(row_of_nnz, weights=is_unigram[X_all.indices], minlength=X_all.shape[0])   # distinct content words
eligible = (n_content >= 2) & (~turns_df["is_backchannel"].to_numpy())
rep_sim = np.full(len(turns_df), np.nan, dtype=np.float32)
rep_of = np.full(len(turns_df), -1, dtype=np.int64)
turn_ids = turns_df["turn_id"].to_numpy()
for (_, _), idx in turns_df.groupby(["transcript_id", "party"], sort=False).indices.items():
    idx = idx[eligible[idx]]
    if len(idx) < 2:
        continue
    S = (X_all[idx] @ X_all[idx].T).toarray()
    for j in range(1, len(idx)):
        k = int(np.argmax(S[j, :j]))
        rep_sim[idx[j]] = S[j, k]
        rep_of[idx[j]] = turn_ids[idx[k]]
turns_df["repeat_sim_max"] = np.round(rep_sim, 4)
turns_df["is_repeat"] = np.nan_to_num(rep_sim, nan=0.0) >= REPEAT_SIM_THRESHOLD
turns_df["repeat_of_turn_id"] = np.where(turns_df["is_repeat"], rep_of, -1)
del X_all, rep_vec, row_of_nnz, cue_matrix, fit_mask
gc.collect()
print(f"repeat detection in {time.perf_counter() - t0:.1f}s")

prev = turns_df.groupby("party")[CUE_COLS + ["is_question", "is_backchannel", "is_repeat"]].mean().T.round(4)
print("\nprevalence of each signal by party (share of turns):")
print(prev.to_string())

lexical features in 31.3s
repeat detection in 31.1s

prevalence of each signal by party (share of turns):
party                    Agent  Customer
cue_hold_request        0.0640    0.0000
cue_hold_return         0.0161    0.0000
cue_apology             0.0460    0.0013
cue_empathy             0.1380    0.0000
cue_refusal             0.0219    0.0000
cue_offer_solution      0.0181    0.0000
cue_agent_escalation    0.0135    0.0000
cue_agent_transfer      0.0037    0.0000
cue_escalation_request  0.0000    0.0016
cue_churn_intent        0.0000    0.0044
cue_refund              0.0554    0.0393
cue_negative_emotion    0.0000    0.0545
cue_confusion           0.0000    0.0136
cue_repeat_contact      0.0000    0.0010
cue_issue_persistence   0.0000    0.0040
cue_competitor_mention  0.0000    0.0119
is_question             0.1794    0.1515
is_backchannel          0.0009    0.0052
is_repeat               0.0012    0.0018


## 10 · Conversation-level table: aggregates, weak event signals, quality flags

In [12]:
t0 = time.perf_counter()
g = turns_df.groupby("transcript_id")
agg = pd.DataFrame({
    "n_turns": g.size(),
    "n_words": g["n_words"].sum(),
    "n_blocks": g["block_id"].max() + 1,
    "n_same_speaker_consecutive": g["same_speaker_as_prev"].sum(),
})
for role in ["Agent", "Customer", "Supervisor", "Specialist", "System"]:
    agg[f"n_turns_{role.lower()}"] = turns_df[turns_df["speaker"] == role].groupby("transcript_id").size().reindex(agg.index, fill_value=0)
agg["n_speakers"] = g["speaker_label_raw"].nunique()

party_cols = ["n_words", "n_filled_pauses", "n_repetitions", "n_hedges", "is_question", "is_backchannel", "is_repeat"] + CUE_COLS
pp = turns_df.groupby(["transcript_id", "party"])[party_cols].sum().unstack("party", fill_value=0)
pp.columns = [f"{party.lower()}_{col}" for col, party in pp.columns]
keep = []
for c in pp.columns:
    party, col = c.split("_", 1)
    scope = CUES[col[4:]][0] if col.startswith("cue_") else "Any"
    if scope in ("Any", party.capitalize()):
        keep.append(c)
pp = pp[keep].reindex(agg.index, fill_value=0)
agg = agg.join(pp)
for party in ("agent", "customer"):
    agg[f"{party}_word_share"] = (agg[f"{party}_n_words"] / agg["n_words"].clip(lower=1)).round(4)
    agg[f"{party}_filled_pause_rate"] = (agg[f"{party}_n_filled_pauses"] / agg[f"{party}_n_words"].clip(lower=1)).round(4)

# non-verbal events: inline markers + marker-only turns
nv_counter = {rec["transcript_id"]: Counter([e["type"] for e in rec["events"]]) for rec in records}
for types in turns_df[["transcript_id", "nonverbal_types"]].itertuples(index=False):
    nv_counter[types.transcript_id].update(types.nonverbal_types)
for kind in ["PAUSE", "HOLD", "TRANSFER", "CALL_END", "NOISE", "SIGH", "LAUGH"]:
    agg[f"n_nonverbal_{kind.lower()}"] = [nv_counter.get(tid, Counter()).get(kind, 0) for tid in agg.index]


def first_turn(mask: pd.Series) -> pd.Series:
    return turns_df.loc[mask].groupby("transcript_id")["turn_id"].min().reindex(agg.index)


esc_first = pd.concat([first_turn(turns_df["speaker"] == "Supervisor"), first_turn(turns_df["cue_escalation_request"]),
                       first_turn(turns_df["cue_agent_escalation"])], axis=1).min(axis=1)
nv_transfer_first = pd.Series({rec["transcript_id"]: min([e["after_turn_id"] + 1 for e in rec["events"] if e["type"] == "TRANSFER"], default=np.nan)
                               for rec in records}).reindex(agg.index)
inline_transfer = first_turn(turns_df["nonverbal_types"].map(lambda v: "TRANSFER" in v))
tr_first = pd.concat([first_turn(turns_df["speaker"] == "Specialist"), first_turn(turns_df["cue_agent_transfer"]),
                      inline_transfer, nv_transfer_first], axis=1).min(axis=1)
signals = pd.DataFrame(index=agg.index)
signals["esc_supervisor_joined"] = agg["n_turns_supervisor"] > 0
signals["esc_customer_requested"] = agg["customer_cue_escalation_request"] > 0
signals["esc_agent_escalated"] = agg["agent_cue_agent_escalation"] > 0
signals["ev_escalation"] = signals[["esc_supervisor_joined", "esc_customer_requested", "esc_agent_escalated"]].any(axis=1)
signals["escalation_turn_id"] = esc_first
signals["ev_transfer"] = tr_first.notna()
signals["transfer_turn_id"] = tr_first
signals["ev_refund"] = (agg.get("agent_cue_refund", 0) + agg.get("customer_cue_refund", 0)) > 0
signals["refund_turn_id"] = first_turn(turns_df["cue_refund"])
signals["ev_churn_risk"] = agg["customer_cue_churn_intent"] > 0
signals["churn_turn_id"] = first_turn(turns_df["cue_churn_intent"])
signals["ev_repeat_contact"] = agg["customer_cue_repeat_contact"] > 0
signals["repeat_contact_turn_id"] = first_turn(turns_df["cue_repeat_contact"])

conv = conv_df.set_index("transcript_id").join(agg).join(signals)
rec_by_id = {r["transcript_id"]: r for r in records}
conv["split"] = [rec_by_id[t]["split"] for t in conv.index]
conv["dedup_group"] = [rec_by_id[t]["dedup_group"] for t in conv.index]
conv["near_dup_cluster_size"] = [rec_by_id[t]["near_dup_cluster_size"] for t in conv.index]
conv["template_group"] = [rec_by_id[t]["template_group"] for t in conv.index]
conv["template_group_size"] = [rec_by_id[t]["template_group_size"] for t in conv.index]
conv["n_annotations_removed"] = [rec_by_id[t]["n_annotations_removed"] for t in conv.index]
conv["n_marker_events"] = [len(rec_by_id[t]["events"]) for t in conv.index]
conv["n_raw_turns"] = [len(rec_by_id[t]["raw_turns"]) for t in conv.index]
conv["had_embedded_dialogue"] = [rec_by_id[t]["had_embedded_dialogue"] for t in conv.index]
conv["had_named_speakers"] = [any(v["method"] != "label_rule" for v in rec_by_id[t]["speaker_map"].values()) for t in conv.index]
low_conf_ids = set(role_map_df.loc[role_map_df["low_confidence"], "transcript_id"])
conv["role_low_confidence"] = [t in low_conf_ids for t in conv.index]
# calls without any turn have no aggregates: counts -> 0, events -> False, event turn ids -> -1 (= absent)
for c in agg.columns:
    conv[c] = conv[c].fillna(0)
for c in [c for c in conv.columns if c.startswith(("n_", "agent_n_", "customer_n_", "agent_is_", "customer_is_", "agent_cue_", "customer_cue_"))]:
    conv[c] = conv[c].astype(int)
for c in [c for c in conv.columns if c.startswith(("esc_", "ev_"))]:
    conv[c] = conv[c].fillna(False).astype(bool)
for c in [c for c in conv.columns if c.endswith("_turn_id")]:
    conv[c] = conv[c].fillna(-1).astype(int)

CLOSING_RE = re.compile(r"\b(?:bye|goodbye|good bye|have a (?:great|good|nice|wonderful|lovely|safe) (?:day|one|evening|night|weekend|trip|flight|stay)|"
                        r"take care|thank(?:s| you) for (?:calling|contacting|choosing|your patience|your time|reaching out)|anything else|"
                        r"you're welcome|that's all|that is all|appreciate (?:it|your help))\b")
last3 = turns_df.groupby("transcript_id").tail(3).groupby("transcript_id")["text_clean"].agg(" ".join)
conv["qc_possibly_truncated"] = [not CLOSING_RE.search(last3.get(t, "")) for t in conv.index]
conv["qc_empty"] = conv["n_turns"] == 0
conv["qc_short"] = conv["n_turns"] < MIN_TURNS_OK
conv["qc_no_customer"] = conv["n_turns_customer"] == 0
conv["qc_no_agent"] = (conv["n_turns"] - conv["n_turns_customer"]) == 0
conv["qc_single_party"] = conv["qc_no_customer"] | conv["qc_no_agent"]
conv["qc_missing_metadata"] = conv["intent"].isna() | conv["reason_for_call"].isna() | conv["time_of_interaction"].isna()
conv["qc_ok"] = ~(conv["qc_empty"] | conv["qc_short"] | conv["qc_single_party"])
conv = conv.reset_index().rename(columns={"index": "transcript_id"}).sort_values("file_order").reset_index(drop=True)

# denormalise the most useful call metadata onto the turn table for filtering
turns_df = turns_df.merge(conv[["transcript_id", "business_domain", "intent_key", "split", "qc_ok"]], on="transcript_id", how="left")
print(f"conversation table: {conv.shape} ({time.perf_counter() - t0:.1f}s)")
sig_cols = ["ev_escalation", "esc_supervisor_joined", "esc_customer_requested", "esc_agent_escalated", "ev_transfer",
            "ev_refund", "ev_churn_risk", "ev_repeat_contact"]
print("\nweak event signal prevalence (share of calls):")
print(conv[sig_cols].mean().round(4).to_string())
print("\nby business domain:")
print(conv.groupby("business_domain")[["ev_escalation", "ev_refund", "ev_churn_risk", "ev_repeat_contact"]].mean().round(3).to_string())
print("\nquality flags:", conv[[c for c in conv.columns if c.startswith("qc_")]].sum().to_dict())
REPORT["signal_prevalence"] = conv[sig_cols].mean().round(4).to_dict()
REPORT["quality_flags"] = {k: int(v) for k, v in conv[[c for c in conv.columns if c.startswith("qc_")]].sum().items()}

conversation table: (19621, 102) (3.3s)

weak event signal prevalence (share of calls):
ev_escalation             0.1602
esc_supervisor_joined     0.0083
esc_customer_requested    0.0256
esc_agent_escalated       0.1532
ev_transfer               0.0594
ev_refund                 0.3405
ev_churn_risk             0.0724
ev_repeat_contact         0.0173

by business domain:
                  ev_escalation  ev_refund  ev_churn_risk  ev_repeat_contact
business_domain                                                             
airline                   0.188      0.720          0.039              0.014
banking                   0.187      0.262          0.058              0.027
hospitality               0.169      0.248          0.072              0.016
insurance                 0.119      0.101          0.068              0.009
retail_ecommerce          0.186      0.743          0.021              0.020
telecom                   0.126      0.024          0.186              0.023

quality fl

### 10.1 Sanity check of the weak signals against the quarantined analyst notes

In [13]:
if len(quar_df):
    chk = quar_df.merge(conv[["transcript_id", "ev_escalation", "ev_churn_risk", "ev_refund"]], on="transcript_id", how="inner")
    rules = {"ev_escalation": (r"\bno escalation\b", r"\bescalat"), "ev_churn_risk": (r"\bno churn\b", r"\bchurn"),
             "ev_refund": (r"\bno refund\b", r"\brefund")}
    rows_chk = []
    for col, (neg, posit) in rules.items():
        for _, r in chk.iterrows():
            txt = r["text"]
            if re.search(neg, txt):
                rows_chk.append({"signal": col, "note_says": False, "signal_says": bool(r[col]), "note": txt[:100]})
            elif re.search(posit, txt):
                rows_chk.append({"signal": col, "note_says": True, "signal_says": bool(r[col]), "note": txt[:100]})
    chk_df = pd.DataFrame(rows_chk, columns=["signal", "note_says", "signal_says", "note"]).drop_duplicates()
    if len(chk_df):
        chk_df["agree"] = chk_df["note_says"] == chk_df["signal_says"]
        print(chk_df.groupby("signal")["agree"].agg(["mean", "size"]).to_string())
        REPORT["note_agreement"] = chk_df.groupby("signal")["agree"].mean().round(3).to_dict()
    else:
        print("no outcome statements found in the quarantined notes")

                   mean  size
signal                       
ev_churn_risk  1.000000     1
ev_escalation  0.333333     3
ev_refund      1.000000     3


### 10.2 Metadata leakage audit: which fields already "know" the outcome?

In [14]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_predict
from sklearn.preprocessing import OneHotEncoder

OUTCOME_LANGUAGE_RE = re.compile(
    r"\b(?:the )?(?:agent|representative|rep|advisor|supervisor|manager|specialist)s? (?:\w+ ){0,2}?"
    r"(?:resolved|escalated|transferred|refunded|offered|issued|processed|apologi[sz]ed|agreed|confirmed|refused|"
    r"declined|waived|credited|assured|rebooked|replaced|reversed|approved)\b"
    r"|\b(?:was|were|got|been) (?:escalated|transferred) to\b"
    r"|\b(?:during|by the end of|at the end of|later in|towards the end of) the (?:call|conversation)\b"
    r"|\bthe (?:call|conversation) (?:was |got )?(?:resolved|escalated|ended|concluded)\b")
assert OUTCOME_LANGUAGE_RE.search("the agent processed a refund at the end of the call")
assert OUTCOME_LANGUAGE_RE.search("customer was escalated to a supervisor")
assert not OUTCOME_LANGUAGE_RE.search("customer called because the flight was cancelled and the claim was denied")
assert not OUTCOME_LANGUAGE_RE.search("wants to speak to a supervisor about a denied claim")

AUDIT_FIELDS = {"reason_for_call": "text", "intent_key": "category", "business_domain": "category"}
AUDIT_TARGETS = ["ev_escalation", "ev_refund", "ev_churn_risk", "ev_repeat_contact", "ev_transfer"]
DERIVED_FROM = {   # columns computed from audited fields inherit their flag
    "intent": ["intent_key"], "domain_keyword_vote": ["reason_for_call", "intent_key"],
    "domain_keyword_confidence": ["reason_for_call", "intent_key"], "business_domain": ["reason_for_call", "intent_key"],
    "business_domain_source": ["business_domain"]}

t0 = time.perf_counter()
aud = conv[(conv["split"] != "test") & conv["qc_ok"]].reset_index(drop=True)
opening_text = turns_df[turns_df["turn_id"] < OPENING_TURNS].groupby("transcript_id")["text_clean"].agg(" ".join)


def audit_matrix(kind: str, values):
    if kind == "text":
        vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100_000, sublinear_tf=True, dtype=np.float32)
        return vec.fit_transform(values)
    return OneHotEncoder(handle_unknown="ignore").fit_transform(values)


def hashed_group_folds(groups, k: int = 5) -> list:
    f = np.array([int(hashlib.sha1(f"{SPLIT_SALT}:fold:{g}".encode("utf-8")).hexdigest()[:8], 16) % k for g in groups])
    return [(np.flatnonzero(f != j), np.flatnonzero(f == j)) for j in range(k) if 0 < (f == j).sum() < len(f)]


def grouped_cv_auc(X, y, folds) -> float:
    try:
        clf = LogisticRegression(C=1.0, max_iter=1000, solver="liblinear", random_state=SEED)
        p = cross_val_predict(clf, X, y, cv=folds, method="predict_proba")[:, 1]
        return float(roc_auc_score(y, p))
    except ValueError:      # a fold without both classes, ...
        return float("nan")


# ---- content probe ----
content_rows = []
for field in ("reason_for_call", "intent"):
    vals = aud[field].dropna().astype(str).str.lower()
    hit = vals.map(lambda v: bool(OUTCOME_LANGUAGE_RE.search(v)))
    share = float(hit.mean()) if len(vals) else 0.0
    content_rows.append({"probe": "content", "field": field, "n_calls": int(len(vals)),
                         "unique_share": round(vals.nunique() / max(len(vals), 1), 4),
                         "outcome_language_share": round(share, 4), "flagged": share >= LEAKAGE_CONTENT_SHARE,
                         "examples": " || ".join(v[:160] for v in vals[hit].head(3))})

# ---- predictive probe ----
MATRICES, pred_rows = {}, []
try:
    MATRICES["opening_turns"] = audit_matrix("text", aud["transcript_id"].map(opening_text).fillna(""))
except ValueError:
    MATRICES["opening_turns"] = None
for field, kind in AUDIT_FIELDS.items():
    values = aud[field].fillna("").astype(str).str.lower() if kind == "text" else aud[[field]].fillna("unknown").astype(str)
    try:
        MATRICES[field] = audit_matrix(kind, values)
    except ValueError:      
        MATRICES[field] = None
groups = aud["template_group"].to_numpy()
folds = hashed_group_folds(groups)
for target in AUDIT_TARGETS:
    y = aud[target].astype(int).to_numpy()
    n_pos = int(y.sum())
    if min(n_pos, len(y) - n_pos) < 20 or len(folds) < 5:
        pred_rows.append({"probe": "predictive", "field": "(all)", "target": target, "n_calls": int(len(y)),
                          "n_events": n_pos, "note": "skipped: fewer than 20 calls in a class or fewer than 5 folds"})
        continue
    auc_open = grouped_cv_auc(MATRICES["opening_turns"], y, folds) if MATRICES["opening_turns"] is not None else float("nan")
    for field in AUDIT_FIELDS:
        auc = grouped_cv_auc(MATRICES[field], y, folds) if MATRICES[field] is not None else float("nan")
        excess = auc - auc_open
        pred_rows.append({"probe": "predictive", "field": field, "target": target, "n_calls": int(len(y)),
                          "n_events": n_pos, "auc_field": round(auc, 4), "auc_opening": round(auc_open, 4),
                          "excess_auc": round(excess, 4),
                          "flagged": bool(auc >= LEAKAGE_AUC_FLAG and excess >= LEAKAGE_EXCESS_AUC)})
leak_df = pd.DataFrame(content_rows + pred_rows, columns=[
    "probe", "field", "target", "n_calls", "n_events", "unique_share", "outcome_language_share", "auc_field",
    "auc_opening", "excess_auc", "flagged", "examples", "note"])
leak_df["flagged"] = leak_df["flagged"].fillna(False).astype(bool)
leak_df[["n_calls", "n_events"]] = leak_df[["n_calls", "n_events"]].astype("Int64")

LEAKY_FIELDS = set(leak_df.loc[leak_df["flagged"], "field"])
if LEAKY_FIELDS & {"intent", "intent_key"}:
    LEAKY_FIELDS |= {"intent", "intent_key"}
for col, parents in DERIVED_FROM.items():
    if any(p in LEAKY_FIELDS for p in parents):
        LEAKY_FIELDS.add(col)
print(f"leakage audit on {len(aud):,} non-test calls ({time.perf_counter() - t0:.1f}s)")
print(leak_df[leak_df["probe"] == "content"][["field", "n_calls", "unique_share", "outcome_language_share", "flagged"]].to_string(index=False))
pv = leak_df[(leak_df["probe"] == "predictive") & (leak_df["field"] != "(all)")]
if len(pv):
    print(pv.pivot_table(index="field", columns="target", values="auc_field").round(3).to_string())
    print("opening-turns reference AUC:", pv.groupby("target")["auc_opening"].first().round(3).to_dict())
print("FLAGGED (not usable as pre-treatment variables or cohort filters):", sorted(LEAKY_FIELDS) or "none")
REPORT["leakage_audit"] = {"flagged_fields": sorted(LEAKY_FIELDS), "n_calls": int(len(aud)),
                           "rows": leak_df.drop(columns=["examples"]).to_dict("records")}
del MATRICES
gc.collect()

leakage audit on 17,664 non-test calls (10.2s)
          field  n_calls  unique_share  outcome_language_share  flagged
reason_for_call    17664        0.9986                     0.0    False
         intent    17664        0.0020                     0.0    False
target           ev_churn_risk  ev_escalation  ev_refund  ev_repeat_contact  ev_transfer
field                                                                                   
business_domain          0.668          0.546      0.828              0.579        0.540
intent_key               0.720          0.706      0.928              0.672        0.676
reason_for_call          0.722          0.704      0.941              0.683        0.650
opening-turns reference AUC: {'ev_churn_risk': 0.72, 'ev_escalation': 0.69, 'ev_refund': 0.934, 'ev_repeat_contact': 0.668, 'ev_transfer': 0.639}
FLAGGED (not usable as pre-treatment variables or cohort filters): none


460

### 10.3 Column-availability registry: *when* is each column known?

In [15]:
TURN_COLS = ["transcript_id", "turn_id", "turn_uid", "speaker", "party", "speaker_name", "speaker_label_raw", "role_method",
             "text", "text_clean", "text_raw", "block_id", "same_speaker_as_prev", "party_turn_index", "rel_position", "phase",
             "orig_index", "sub_index", "from_embedded", "nonverbal_types", "nonverbal_raw", "n_words", "n_chars",
             "n_filled_pauses", "filled_pause_rate", "n_repetitions", "n_hedges", "n_discourse_markers", "is_question",
             "is_backchannel", "repeat_sim_max", "is_repeat", "repeat_of_turn_id"] + CUE_COLS + ["cues", "business_domain",
             "intent_key", "split", "qc_ok"]
DESIGN_COLS = {"transcript_id", "turn_id", "turn_uid", "orig_index", "sub_index", "from_embedded", "file_order", "split",
               "dedup_group", "near_dup_cluster_size", "template_group", "template_group_size"}
OPENING_COLS = {"brand", "opening_domain_vote", "opening_domain_confidence"}
METADATA_COLS = {"time_of_interaction", "call_date", "call_hour", "call_weekday", "domain_raw", "intent", "intent_key",
                 "reason_for_call", "business_domain", "business_domain_source", "domain_keyword_vote",
                 "domain_keyword_confidence", "extra_meta_json"}
TURN_PAST_COLS = {"speaker", "party", "speaker_name", "speaker_label_raw", "role_method", "text", "text_clean", "text_raw",
                  "block_id", "same_speaker_as_prev", "party_turn_index", "nonverbal_types", "nonverbal_raw", "n_words",
                  "n_chars", "n_filled_pauses", "filled_pause_rate", "n_repetitions", "n_hedges", "n_discourse_markers",
                  "is_question", "is_backchannel", "repeat_sim_max", "is_repeat", "repeat_of_turn_id", "cues"} | set(CUE_COLS)
TURN_WHOLE_CALL_COLS = {"rel_position", "phase", "qc_ok"}
CALL_WHOLE_CALL_COLS = {"role_low_confidence", "had_embedded_dialogue", "had_named_speakers"}
USE_RULE = {"design": "no (bookkeeping / clustering unit)", "turn-past": "yes, for turns before the landmark",
            "opening": "yes, for landmarks at or after OPENING_TURNS", "whole-call": "no (uses turns after the landmark)",
            "outcome": "no (outcome only)"}


def availability(col: str, table: str):
    if col in DESIGN_COLS:
        return "design"
    if col in METADATA_COLS:
        return "metadata"
    if table == "turns":
        return "turn-past" if col in TURN_PAST_COLS else ("whole-call" if col in TURN_WHOLE_CALL_COLS else None)
    if col in OPENING_COLS:
        return "opening"
    if col.startswith(("ev_", "esc_")) or col.endswith("_turn_id"):
        return "outcome"
    if col in CALL_WHOLE_CALL_COLS or col.startswith(("n_", "agent_", "customer_", "qc_")):
        return "whole-call"
    return None


avail_rows = []
for table, cols in (("turns", TURN_COLS), ("conversations", list(conv.columns))):
    for col in cols:
        cls = availability(col, table)
        flag = col in LEAKY_FIELDS
        rule = USE_RULE.get(cls, "") if cls != "metadata" else (
            "no (flagged by the leakage audit)" if flag else "sensitivity analyses only (timing unknown)")
        avail_rows.append({"table": table, "column": col, "availability": cls, "leakage_flag": flag,
                           "usable_as_covariate": rule})
avail_df = pd.DataFrame(avail_rows)
print(avail_df.groupby(["table", "availability"], dropna=False).size().rename("columns").to_string())
REPORT["column_availability"] = avail_df.groupby("availability", dropna=False).size().to_dict()

table          availability
conversations  design           7
               metadata        12
               opening          3
               outcome         13
               whole-call      67
turns          design           7
               metadata         2
               turn-past       42
               whole-call       3


## 11 · Integrity checks

In [16]:
CHECKS = []


def check(name: str, ok: bool, detail: str = "") -> None:
    CHECKS.append({"check": name, "ok": bool(ok), "detail": detail})


tt = turns_df
check("transcript_id unique in conversations", conv["transcript_id"].is_unique)
check("turn_uid unique", tt["turn_uid"].is_unique)
contig = tt.groupby("transcript_id")["turn_id"].agg(lambda s: bool((s.to_numpy() == np.arange(len(s))).all()))
check("turn_id contiguous 0..n-1 per call", bool(contig.all()), f"{int((~contig).sum())} bad")
check("n_turns matches turn table", bool((conv.set_index("transcript_id")["n_turns"].reindex(contig.index) == tt.groupby("transcript_id").size()).all()))
check("every turn belongs to a known call", bool(tt["transcript_id"].isin(conv["transcript_id"]).all()))
check("speaker in canonical set", bool(tt["speaker"].isin(ROLE_DISPLAY.values()).all()))
check("party is Agent/Customer", bool(tt["party"].isin(["Agent", "Customer"]).all()))
check("no empty text", bool((tt["text"].str.len() > 0).all() and (tt["text_clean"].str.len() > 0).all()))
check("no literal or real newlines left", not bool(tt["text"].str.contains(r"\\n|\n", regex=True).any()))
check("no annotation labels left among speakers",
      not any(classify_label(l)["kind"] == "annotation" for l in tt["speaker_label_raw"].unique()))
expected_turns = struct["turns_after_structural_pass"] - REPORT["duplicates"]["turns_in_dropped_duplicates"]
check("turn accounting: raw + split - quarantined - markers - empty - duplicates = output",
      len(tt) == expected_turns, f"{len(tt)} vs {expected_turns}")
check("splits cover all calls", bool(conv["split"].isin(["train", "dev", "test"]).all()))
check("near-dup clusters never straddle splits", bool(conv.groupby("dedup_group")["split"].nunique().max() == 1))
check("template groups never straddle splits", bool(conv.groupby("template_group")["split"].nunique().max() == 1))
check("near-dup clusters nested in template groups", bool(conv.groupby("dedup_group")["template_group"].nunique().max() == 1))
unclassified = avail_df.loc[avail_df["availability"].isna(), "column"].tolist()
check("every output column has an availability class", not unclassified, ", ".join(unclassified))
check("leakage-flagged columns are never usable as covariates",
      bool((~avail_df["leakage_flag"] | avail_df["usable_as_covariate"].str.startswith("no")).all()))
event_cols = [c for c in conv.columns if c.endswith("_turn_id")]
check("event turn ids are -1 (absent) or point to an existing turn",
      bool(all(((conv[c] >= -1) & (conv[c] < conv["n_turns"])).all() for c in event_cols)), ", ".join(event_cols))
check("event flag <=> event turn id", bool((conv["ev_escalation"] == (conv["escalation_turn_id"] >= 0)).all()
                                             and (conv["ev_refund"] == (conv["refund_turn_id"] >= 0)).all()
                                             and (conv["ev_churn_risk"] == (conv["churn_turn_id"] >= 0)).all()))
checks_df = pd.DataFrame(CHECKS)
print(checks_df.to_string(index=False))
REPORT["integrity_checks"] = CHECKS
assert checks_df["ok"].all(), "integrity check failed — see table above"

                                                                             check   ok                                                                                      detail
                                             transcript_id unique in conversations True                                                                                            
                                                                   turn_uid unique True                                                                                            
                                                turn_id contiguous 0..n-1 per call True                                                                                       0 bad
                                                        n_turns matches turn table True                                                                                            
                                                every turn belongs to a known call True             

## 12 · Save `cleaned_data/`

In [17]:
def to_py(v):
    if isinstance(v, dict):
        return {str(k): to_py(x) for k, x in v.items()}
    if isinstance(v, (list, tuple, np.ndarray)):
        return [to_py(x) for x in v]
    if isinstance(v, np.bool_):
        return bool(v)
    if isinstance(v, np.integer):
        return int(v)
    if isinstance(v, (float, np.floating)):
        return None if math.isnan(v) else float(v)
    if v is None or v is pd.NaT or v is pd.NA:
        return None
    return v


def save_table(df: pd.DataFrame, name: str) -> str:
    try:
        df.to_parquet(OUT_DIR / f"{name}.parquet", index=False)
        return f"{name}.parquet"
    except Exception as exc: 
        print(f"parquet unavailable ({exc.__class__.__name__}); writing CSV for {name}")
        out = df.copy()
        for c in out.columns:
            if out[c].map(lambda v: isinstance(v, (list, dict))).any():
                out[c] = out[c].map(lambda v: json.dumps(to_py(v)))
        out.to_csv(OUT_DIR / f"{name}.csv", index=False)
        return f"{name}.csv"


t0 = time.perf_counter()
turns_out = turns_df[TURN_COLS]      
files = {"turns": save_table(turns_out, "turns"), "conversations": save_table(conv, "conversations")}
quar_out = quar_df.merge(conv[["transcript_id", "split"]], on="transcript_id", how="left")
files["annotations_quarantine"] = save_table(quar_out, "annotations_quarantine")
role_map_df.to_csv(OUT_DIR / "speaker_role_map.csv", index=False)
domain_map_df.to_csv(OUT_DIR / "domain_mapping.csv", index=False)
dups_df.to_csv(OUT_DIR / "duplicates.csv", index=False)
leak_df.to_csv(OUT_DIR / "leakage_audit.csv", index=False)
avail_df.to_csv(OUT_DIR / "column_availability.csv", index=False)
pd.DataFrame(QC_LOG, columns=["transcript_id", "stage", "issue", "detail"]).to_csv(OUT_DIR / "qc_log.csv", index=False)

META_COLS = ["time_of_interaction", "call_date", "call_hour", "call_weekday", "domain_raw", "business_domain",
             "business_domain_source", "opening_domain_vote", "intent", "intent_key", "reason_for_call", "brand", "split",
             "template_group"]
SIGNAL_COLS = ["ev_escalation", "escalation_turn_id", "esc_supervisor_joined", "esc_customer_requested", "esc_agent_escalated",
               "ev_transfer", "transfer_turn_id", "ev_refund", "refund_turn_id", "ev_churn_risk", "churn_turn_id",
               "ev_repeat_contact", "repeat_contact_turn_id"]
QUALITY_COLS = [c for c in conv.columns if c.startswith("qc_")] + ["role_low_confidence", "had_embedded_dialogue",
                "had_named_speakers", "n_annotations_removed", "n_marker_events", "near_dup_cluster_size", "template_group_size",
                "n_raw_turns"]
JSON_TURN_COLS = ["turn_id", "speaker", "party", "speaker_name", "text", "text_clean", "cues", "nonverbal_types"]
turn_groups = defaultdict(list)
for t in turns_out[["transcript_id"] + JSON_TURN_COLS].to_dict("records"):
    turn_groups[t.pop("transcript_id")].append(to_py(t))
out_records = []
for row in conv.to_dict("records"):
    tid = row["transcript_id"]
    turns_json = turn_groups.get(tid, [])
    rec = rec_by_id[tid]
    out_records.append({
        "transcript_id": tid,
        **{k: to_py(row[k]) for k in META_COLS},
        "n_turns": int(row["n_turns"]),
        "speakers": [{"label": v["label"], "role": v["role"], "method": v["method"], "speaker_name": v["speaker_name"]}
                     for v in rec["speaker_map"].values()],
        "signals": {k: to_py(row[k]) for k in SIGNAL_COLS},
        "quality": {k: to_py(row[k]) for k in QUALITY_COLS},
        "events": [to_py(e) for e in rec["events"]],
        "conversation": turns_json,
    })
with open(OUT_DIR / "cleaned_data.json", "w", encoding="utf-8") as f:
    json.dump(out_records, f, ensure_ascii=False, separators=(",", ":"))
files["cleaned_data"] = "cleaned_data.json"
print(f"saved in {time.perf_counter() - t0:.1f}s ->", {k: v for k, v in files.items()})

saved in 22.5s -> {'turns': 'turns.parquet', 'conversations': 'conversations.parquet', 'annotations_quarantine': 'annotations_quarantine.parquet', 'cleaned_data': 'cleaned_data.json'}


### 12.1 Cleaning report & data card (datasheet)

In [19]:
COLUMN_DOCS = {
    # turns
    "turn_id": "0-based position of the turn in the cleaned call (citeable id)",
    "turn_uid": "globally unique turn key `<transcript_id>:<turn_id>` for evidence citation",
    "speaker": "canonical role: Agent | Customer | Supervisor | Specialist | System",
    "party": "two-party view: Customer vs Agent (all company-side roles)",
    "speaker_name": "person name when the raw label was a name (else null)",
    "speaker_label_raw": "speaker label exactly as in the raw file (or the label recovered from an embedded dialogue)",
    "role_method": "how the role was obtained: label_rule | classifier_* | context_rule",
    "text": "normalised verbatim text (NFKC, straight quotes, lower case, no escapes/markers)",
    "text_clean": "`text` without filled pauses and immediate repetitions (for embeddings / NLI / LLMs)",
    "text_raw": "original text of the (split) turn",
    "block_id": "index of the speaker-change block (consecutive turns of one speaker share it)",
    "same_speaker_as_prev": "true if the previous turn has the same speaker",
    "party_turn_index": "k-th turn of this party in the call",
    "rel_position": "turn_id / (n_turns-1) in [0,1] (needs the call length: whole-call, never a causal covariate)",
    "phase": "opening (<=0.2) | middle | closing (>=0.8) by rel_position (whole-call)",
    "orig_index": "index of the source turn in the raw conversation list",
    "sub_index": "piece index when a raw turn was split (embedded dialogue)",
    "from_embedded": "turn recovered from an embedded `\\n 2 customer …` dialogue",
    "nonverbal_types": "typed stage directions in the turn (PAUSE, HOLD, TRANSFER, CALL_END, NOISE, SIGH, LAUGH, TYPING, OTHER)",
    "nonverbal_raw": "the raw marker texts",
    "n_words": "word count of `text`", "n_chars": "character count of `text`",
    "n_filled_pauses": "count of um/uh/erm/hmm/ah (hesitation)", "filled_pause_rate": "n_filled_pauses / n_words",
    "n_repetitions": "immediate word/bigram repetitions (self-repair)", "n_hedges": "hedges (i guess, maybe, not sure, i think …)",
    "n_discourse_markers": "'you know' / 'i mean' count", "is_question": "heuristic question detector (text is unpunctuated)",
    "is_backchannel": "minimal response (uh huh, okay, right …)",
    "repeat_sim_max": "max TF-IDF cosine to an earlier turn of the same party in the call",
    "is_repeat": f"repeat_sim_max >= {REPEAT_SIM_THRESHOLD}", "repeat_of_turn_id": "turn repeated (or -1)",
    "cues": "list of cue names that fired on this turn",
    # conversations
    "transcript_id": "call id (primary key)", "file_order": "position in the raw file",
    "time_of_interaction": "parsed timestamp (YYYY-MM-DD HH:MM:SS)", "call_date": "date", "call_hour": "hour (-1 unknown)",
    "call_weekday": "weekday name", "domain_raw": "original `domain` field (constant 'Insurance' in the raw data)",
    "business_domain": "inferred industry: hospitality | airline | telecom | banking | insurance | retail_ecommerce | unknown",
    "business_domain_source": "intent_majority | conversation_keywords | manual_override | intent_majority_fallback",
    "domain_keyword_vote": "per-call lexicon vote (intent + reason + first turns)",
    "domain_keyword_confidence": "(top-second)/top lexicon score",
    "opening_domain_vote": f"lexicon vote on the first {OPENING_TURNS} turns only (pre-behaviour industry control)",
    "opening_domain_confidence": "(top-second)/top lexicon score of the opening-only vote",
    "intent": "original intent label (whitespace-normalised)", "intent_key": "slug of intent",
    "reason_for_call": "normalised reason text", "brand": f"company name from the agent greeting in the first {OPENING_TURNS} turns (best effort)",
    "extra_meta_json": "unexpected top-level fields of the raw record (JSON), if any",
    "split": "train | dev | test (hash of template_group; locked before any analysis)",
    "dedup_group": "near-duplicate cluster representative id",
    "near_dup_cluster_size": "size of the near-duplicate cluster (1 = unique)",
    "template_group": "template-group representative id (calls re-using the same wording); cluster unit for inference",
    "template_group_size": "calls in the template group (1 = unique)",
    "n_turns": "turns after cleaning", "n_raw_turns": "turns in the raw record", "n_words": "words in the call",
    "n_blocks": "speaker-change blocks", "n_same_speaker_consecutive": "turns that continue the previous speaker",
    "n_speakers": "distinct raw speaker labels kept", "n_annotations_removed": "quarantined annotation turns",
    "n_marker_events": "marker-only turns converted to events", "had_embedded_dialogue": "record needed re-splitting",
    "had_named_speakers": "at least one role came from the classifier/context rules",
    "role_low_confidence": "a classifier-resolved role had margin < ROLE_MIN_MARGIN",
    "ev_escalation": "weak signal: supervisor joined OR customer asked for one OR agent escalated",
    "escalation_turn_id": "first turn of the escalation evidence (-1 none)",
    "esc_supervisor_joined": "a Supervisor speaks", "esc_customer_requested": "customer asked for supervisor/manager",
    "esc_agent_escalated": "agent mentions escalating / supervisor / manager",
    "ev_transfer": "weak signal: specialist joined / transfer phrase / TRANSFER marker", "transfer_turn_id": "first transfer turn (-1 none)",
    "ev_refund": "weak signal: refund/compensation talk", "refund_turn_id": "first refund turn (-1 none)",
    "ev_churn_risk": "weak signal: customer intent to cancel/switch/not renew", "churn_turn_id": "first churn turn (-1 none)",
    "ev_repeat_contact": "weak signal: customer says they already called/explained", "repeat_contact_turn_id": "first such turn (-1 none)",
    "qc_ok": "usable for statistics: not empty/short/single-party", "qc_empty": "no turns", "qc_short": f"< {MIN_TURNS_OK} turns",
    "qc_no_customer": "no customer turn", "qc_no_agent": "no company-side turn", "qc_single_party": "only one party",
    "qc_possibly_truncated": "no closing formula in the last 3 turns", "qc_missing_metadata": "intent/reason/time missing",
    "label_raw": "raw label of the quarantined turn", "reason": "why it was quarantined",
}
for c in CUE_COLS:
    COLUMN_DOCS[c] = f"lexical cue ({CUES[c[4:]][0]} turns): {c[4:].replace('_', ' ')}"
for c in conv.columns:
    if c.startswith(("agent_", "customer_")) and c not in COLUMN_DOCS:
        party, rest = c.split("_", 1)
        COLUMN_DOCS[c] = f"{party} turns: " + ("sum of " + rest if rest.startswith(("n_", "is_", "cue_")) else rest.replace("_", " "))
for c in [c for c in conv.columns if c.startswith("n_turns_") or c.startswith("n_nonverbal_")]:
    COLUMN_DOCS.setdefault(c, c.replace("n_turns_", "turns spoken by ").replace("n_nonverbal_", "non-verbal events of type "))
undocumented = [c for c in list(turns_out.columns) + list(conv.columns) if c not in COLUMN_DOCS]
if undocumented:
    print("undocumented columns:", undocumented)

REPORT.update({
    "generated_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
    "raw_file": str(RAW_PATH.name), "raw_sha256": RAW_SHA256,
    "versions": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
                 "scikit-learn": sklearn.__version__},
    "n_conversations_out": int(len(conv)), "n_turns_out": int(len(turns_out)),
    "n_quarantined": int(len(quar_df)), "files": files, "runtime_seconds": round(time.perf_counter() - T0, 1),
})
with open(OUT_DIR / "cleaning_report.json", "w", encoding="utf-8") as f:
    json.dump(to_py(REPORT), f, indent=2, ensure_ascii=False, default=str)


FILE_DOCS = {
    "cleaned_data": "1 record per call, nested cleaned turns (same shape as the raw file) + signals + quality",
    "turns": "1 row per turn: text views, roles, position, non-verbal events, cues, dynamics features",
    "conversations": "1 row per call: metadata, business domain, aggregates, weak event signals, QC flags, split",
    "annotations_quarantine": "analyst notes removed from dialogues (audit only, never evidence)",
}


def md_table(df: pd.DataFrame) -> str:
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    cell = lambda v: "" if v is None or v is pd.NA or (isinstance(v, float) and math.isnan(v)) else str(v).replace("|", "\\|")
    lines += ["| " + " | ".join(cell(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(lines)


def schema_table(df: pd.DataFrame, table: str) -> str:
    av = avail_df[avail_df["table"] == table].set_index("column")["availability"]
    return md_table(pd.DataFrame({"column": df.columns, "dtype": [str(t) for t in df.dtypes],
                                  "availability": [av.get(c, "") for c in df.columns],
                                  "description": [COLUMN_DOCS.get(c, "") for c in df.columns]}))


card = f"""# Data card — cleaned call-centre transcripts

*Generated by `notebooks/01_data_cleaning.ipynb` on {REPORT['generated_at_utc']} UTC.*
**The underlying data is copyrighted and provided only for the internal hackathon — do not redistribute.**

## Provenance
* raw file: `{REPORT['raw_file']}` — SHA-256 `{RAW_SHA256}`
* raw records: {REPORT['n_raw_records']:,} · raw turns: {REPORT['n_raw_turns']:,}
* output: {len(conv):,} calls · {len(turns_out):,} turns · {len(quar_df):,} quarantined annotation texts
* software: Python {platform.python_version()}, pandas {pd.__version__}, numpy {np.__version__}, scikit-learn {sklearn.__version__}

## Cleaning steps (counts)
```
{json.dumps(to_py(REPORT['structural_pass']), indent=2)}
duplicates: {json.dumps(to_py(REPORT['duplicates']))}
```
Speaker-role resolution (hidden-label validation on held-out correctly-labelled calls):
`{json.dumps(to_py(ROLE_EVAL))}`

## Weak event signals (share of calls) — heuristic, not ground truth
{md_table(conv[sig_cols].mean().round(4).rename('share').reset_index().rename(columns={'index': 'signal'}))}

## Business domains (inferred; raw `domain` is constant)
{md_table(conv['business_domain'].value_counts().rename('calls').reset_index())}

## Files
{md_table(pd.DataFrame([{'file': v, 'content': FILE_DOCS[k]} for k, v in files.items()] + [
    {'file': 'speaker_role_map.csv', 'content': 'per call x raw speaker label: role, method, score, margin'},
    {'file': 'domain_mapping.csv', 'content': 'intent -> business domain votes and purity'},
    {'file': 'duplicates.csv', 'content': 'exact (dropped) and near (kept) duplicates'},
    {'file': 'leakage_audit.csv', 'content': 'content + predictive probes of the metadata fields'},
    {'file': 'column_availability.csv', 'content': 'availability class of every output column and whether it may be a covariate'},
    {'file': 'qc_log.csv', 'content': 'every data-quality event, per call'},
    {'file': 'cleaning_report.json', 'content': 'config, counts, versions, checks'}]))}

## Leakage audit of the metadata (§10.2)
Fields flagged (never use as pre-treatment variables or cohort filters): **{', '.join(sorted(LEAKY_FIELDS)) or 'none'}**

{md_table(leak_df[leak_df['probe'] == 'content'][['field', 'n_calls', 'unique_share', 'outcome_language_share', 'flagged']])}

{md_table(leak_df[leak_df['probe'] == 'predictive'][['field', 'target', 'n_events', 'auc_field', 'auc_opening', 'excess_auc', 'flagged']])}

## Column availability (what is known when; §10.3)
{md_table(pd.DataFrame([{'class': k, 'usable as a covariate at landmark turn k': v} for k, v in {**USE_RULE, 'metadata': 'sensitivity analyses only; never if flagged by the leakage audit'}.items()]))}

## Schema — `turns`
{schema_table(turns_out, 'turns')}

## Schema — `conversations`
{schema_table(conv, 'conversations')}

## Known limitations
* Speaker roles for name-labelled calls are *inferred* (see validation above and `speaker_role_map.csv`).
* Cues and event signals are lexicon-based proxies with moderate recall; validate before drawing causal claims.
* Text is lower-cased and unpunctuated ASR-style output; `is_question` is heuristic.
* `business_domain` is inferred; review `domain_mapping.csv` and use `DOMAIN_OVERRIDES` if needed.
* No timestamps per turn: silences are approximated by hold phrases and PAUSE/HOLD markers.
* Metadata timing is undocumented; the leakage audit can flag a leaky field but cannot prove a field is pre-call.
"""
(OUT_DIR / "DATA_CARD.md").write_text(card, encoding="utf-8")
print("wrote cleaning_report.json and DATA_CARD.md")
print(f"total runtime: {time.perf_counter() - T0:.1f}s")

wrote cleaning_report.json and DATA_CARD.md
total runtime: 319.6s


## 13 · Before / after examples
One example per repaired defect, picked automatically.

In [20]:
def show_turns(tid: str, start: int = 0, n: int = 6) -> None:
    sub = turns_df[(turns_df["transcript_id"] == tid) & (turns_df["turn_id"] >= start)].head(n)
    for t in sub.itertuples():
        name = f" ({t.speaker_name})" if isinstance(t.speaker_name, str) else ""
        extra = f"  {list(t.nonverbal_types)}" if len(t.nonverbal_types) else ""
        cues = f"  cues={list(t.cues)}" if len(t.cues) else ""
        print(f"  {t.turn_id:>3} {t.speaker + name:>28}: {t.text_clean[:110]}{extra}{cues}")


def raw_turns(tid: str, start: int = 0, n: int = 6, width: int = 150) -> None:
    for t in rec_by_id[tid]["raw_turns"][start:start + n]:
        print(f"  {t['orig_index']:>3} {t['speaker'][:28]:>28}: {t['text'][:width]}")


def pick(mask: pd.Series):
    ids = conv.loc[mask, "transcript_id"]
    return ids.iloc[0] if len(ids) else None


print("=" * 120)
tid = pick(conv["had_embedded_dialogue"])
if tid:
    print(f"[1] EMBEDDED DIALOGUE re-split | {tid}\n-- raw --")
    raw_turns(tid, n=1, width=400)
    print("-- cleaned --")
    show_turns(tid)
print("=" * 120)
named_all = role_map_df[role_map_df["method"].str.startswith("classifier")]["transcript_id"]
tid = pick(conv["transcript_id"].isin(named_all))
if tid:
    print(f"[2] PERSON-NAME LABELS -> roles | {tid}")
    display(role_map_df[role_map_df["transcript_id"] == tid][["label", "role", "method", "score", "margin"]])
    print("-- raw --")
    raw_turns(tid, n=4)
    print("-- cleaned --")
    show_turns(tid, n=4)
print("=" * 120)
ctx_ids = role_map_df[role_map_df["method"] == "context_rule"]["transcript_id"]
tid = pick(conv["transcript_id"].isin(ctx_ids))
if tid:
    first_sup = int(turns_df[(turns_df["transcript_id"] == tid) & (turns_df["role_method"] == "context_rule")]["turn_id"].min())
    print(f"[3] THIRD SPEAKER resolved from context (escalation / transfer) | {tid}")
    show_turns(tid, start=max(0, first_sup - 3), n=6)
print("=" * 120)
tid = pick(conv["n_annotations_removed"] > 0)
if tid:
    print(f"[4] ANALYST NOTES quarantined (never used as evidence) | {tid}")
    display(quar_df[quar_df["transcript_id"] == tid])
print("=" * 120)
tid = pick(conv["n_marker_events"] > 0)
if tid:
    print(f"[5] MARKER-ONLY TURNS -> events | {tid}")
    for e in rec_by_id[tid]["events"]:
        print("  ", e)

[1] EMBEDDED DIALOGUE re-split | 4e9f931b-c7f0-4798-bfc5-2554d248ab4a
-- raw --
    0                        Agent: thank you for calling aerolink this is lisa chen uh how can i assist you today \n 2 customer hi uh lisa uh this is carter watson i was um calling about a fare change complaint i i have \n 3 agent of course uh carter so can you tell me more about the issue uh about the fare change \n 4 customer well i uh recently noticed a fare change for my flight uh from boston to los angeles and uh i just um don
-- cleaned --
    0                        Agent: thank you for calling aerolink this is lisa chen how can i assist you today
    1                     Customer: hi lisa this is carter watson i was calling about a fare change complaint i have
    2                        Agent: of course carter so can you tell me more about the issue about the fare change
    3                     Customer: well i recently noticed a fare change for my flight from boston to los angeles and i just

,label,role,method,score,margin
48,Chloe Adams,agent,classifier_rank,76.032,134.499
49,Olivia Davis,customer,classifier_rank,-58.467,134.499


-- raw --
    0                  Chloe Adams: thank you for calling beachfront suites this is chloe adams how can i assist you today
    1                 Olivia Davis: hi chloe i’m just looking for some clarity on the loyalty perks um and potential upgrades you know
    2                  Chloe Adams: of course olivia can you tell me if you’re currently um part of a loyalty tier
    3                 Olivia Davis: yeah i’m at the gold tier um does that help me uh with upgrades
-- cleaned --
    0          Agent (Chloe Adams): thank you for calling beachfront suites this is chloe adams how can i assist you today
    1      Customer (Olivia Davis): hi chloe i'm just looking for some clarity on the loyalty perks and potential upgrades you know
    2          Agent (Chloe Adams): of course olivia can you tell me if you're currently part of a loyalty tier
    3      Customer (Olivia Davis): yeah i'm at the gold tier does that help me with upgrades
[3] THIRD SPEAKER resolved from context (e

,transcript_id,orig_index,label_raw,text,reason
0,f6496ee6-8477-4ee4-a48d-284a421e5ca4,40,Domain,insurance,annotation_label
1,f6496ee6-8477-4ee4-a48d-284a421e5ca4,41,Domain Category,cancellation policies,annotation_label


[5] MARKER-ONLY TURNS -> events | 775eebb2-be5a-42ea-9d36-99767ea03d6b
   {'after_turn_id': 37, 'orig_index': 38, 'label_raw': 'Agent', 'type': 'CALL_END', 'raw': 'call ended'}
